# Data Augmentation per la Sicurezza delle Centrali Elettriche

> **Cliente**: CyberEye Solutions — sicurezza cibernetica per infrastrutture critiche
> **Problema**: il sistema di sorveglianza video delle centrali elettriche si basa su un modello di riconoscimento immagini addestrato su un dataset troppo piccolo e poco vario, che di conseguenza non generalizza bene a situazioni reali
> **Soluzione proposta**: usare una pipeline generativa (captioning → generazione testuale → generazione di immagini) per espandere il dataset di training con almeno il 20% di immagini sintetiche, e verificare se questo migliora davvero le prestazioni del classificatore
> **Dataset usato come base**: Oxford-IIIT Pet (da `torchvision`) — un dataset di immagini di 37 razze di cani e gatti, scelto come sostituto didattico realistico di un dataset di sorveglianza.

## Contesto

Il testo del progetto chiede esplicitamente di lavorare su `OxfordIIITPet`. Non è un dataset di centrali elettriche, ma lo possiamo considerare come un ottimo banco di prova perché condivide i problemi tipici di un caso reale di sorveglianza:

- **Classi visivamente simili tra loro** (alcune razze di gatto si distinguono a fatica, così come due comportamenti "quasi normali" e "quasi pericolosi" in un video di sorveglianza) — un problema di classificazione fine-grained;
- **Necessità di un dataset di training sempre più vario**, dato che in un contesto reale raccogliere ogni possibile variazione di scena, illuminazione e comportamento è costoso o impossibile per gli eventi rari, che sono proprio quelli più critici;
- **Necessità di un modello affidabile**, che è esattamente il problema che la Data Augmentation generativa prova ad aiutare a risolvere, ampliando artificialmente la varietà dei dati disponibili.

## Obiettivo del notebook

1. Costruire il dataset di lavoro a partire da Oxford-IIIT Pet, suddividendolo in training, validation e test.
2. Addestrare un classificatore **baseline** solo sui dati reali di training.
3. Costruire una pipeline di **Data Augmentation generativa** in tre passaggi, applicata a un campione stratificato pari ad almeno il 20% del training:
   - **Image captioning**: si genera una descrizione testuale di ogni immagine campionata;
   - **Generazione testuale**: si generano varianti della descrizione (parafrasi);
   - **Generazione di immagini**: a partire dalle varianti testuali si generano nuove immagini sintetiche con un modello di diffusione.
4. Addestrare un secondo classificatore, identico al primo per architettura e iperparametri, sul training reale **+ almeno il 20% di immagini sintetiche**.
5. Confrontare le prestazioni dei due modelli sullo stesso test set reale, con più metriche, per capire se e quanto la Data Augmentation generativa aiuta davvero.

## Metodologia

Per rendere il confronto tra **Baseline** e **Augmented** corretto, i due esperimenti utilizzano **la stessa struttura e gli stessi parametri di addestramento**: stessa architettura del modello, stesso ottimizzatore, stessi learning rate (differenziati tra backbone e testa), stesso numero massimo di epoche e stesso validation set.

**L'unica differenza riguarda quindi i dati di training:** la Baseline utilizza solo le immagini reali del training set, mentre l'Augmented utilizza le stesse immagini reali più le immagini sintetiche generate.
In questo modo, se le prestazioni di uno dei due modelli risultano migliori, il miglioramento può essere attribuito all'aggiunta dei dati sintetici, e non a differenze nell'architettura o nei parametri di training.

## Struttura del notebook

1. Setup dell'ambiente Colab e installazione librerie
2. Import delle librerie, seed e device
3. Configurazione centralizzata (`Config`)
4. Caricamento ed esplorazione di Oxford-IIIT Pet
5. Costruzione degli split: training, validation, test (`train_test_split` sull'intero trainval)
6. Trasformazioni e DataLoader
7. Architettura del classificatore e classi di supporto (`build_model`, `Trainer`, `Evaluator`)
8. Esperimento A — Baseline (solo dati reali)
9. Valutazione dell'esperimento Baseline sul test set
10. Pipeline di Data Augmentation generativa (campionamento del 30% → captioning → testo → immagini)
11. Valutazione della qualità dei dati sintetici
12. Esperimento B — Augmented (dati reali + sintetici)
13. Valutazione dell'esperimento Augmented sul test set
14. Confronto finale tra Baseline e Augmented
15. Error Analysis (casi corretti/rotti dall'Augmented rispetto al Baseline)
16. Conclusioni, limiti e sviluppi futuri

## 1. Setup dell'ambiente Colab

**Prima di eseguire** selezionare una **GPU** (T4 è sufficiente). 

La cella seguente installa le librerie che Colab non ha già preinstallate o che richiedono una versione più recente:

- `diffusers` e `accelerate`: pipeline di generazione immagini (sdxl-turbo);
- `transformers`: modelli di captioning (BLIP), generazione testuale (Qwen2.5-1.5B-Instruct) e CLIP per la valutazione della qualità dei sintetici;
- `timm`: raccolta di backbone pretrained per PyTorch (qui EfficientNet-B0), con un'interfaccia semplice per sostituire la testa di classificazione;
- `sentencepiece`: tokenizer richiesto da alcuni dei modelli usati (es. CLIP).

In [ ]:
!pip install -q diffusers accelerate transformers timm sentencepiece

## 2. Import delle librerie, seed e device

In [ ]:
# ============================================================
# Import delle librerie
# ============================================================
# Suddivisi per area funzionale:
#
# - os, random, time, copy, gc, dataclasses: utility standard (gc serve per
#   liberare esplicitamente la memoria GPU tra una fase generativa e l'altra)
# - numpy: campionamento stratificato, bootstrap, calcoli su array
# - torch / torchvision: dataset Oxford-IIIT Pet, training del classificatore
# - timm: backbone pretrained (EfficientNet-B0) per il transfer learning
# - transformers: BLIP (captioning), Qwen2.5 (generazione testo), CLIP (valutazione)
# - diffusers: pipeline di generazione immagini (sdxl-turbo)
# - sklearn.metrics: accuracy, precision/recall/F1, top-k accuracy, confusion matrix
# - matplotlib / seaborn: grafici
# - tqdm: barre di avanzamento per i cicli lunghi (captioning, generazione immagini)
# ============================================================
import os
import random
import time
import copy
import gc
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, ConcatDataset, DataLoader
import torchvision
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

import timm

from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    AutoTokenizer, AutoModelForCausalLM,
    CLIPModel, CLIPProcessor,
)
from diffusers import AutoPipelineForText2Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, top_k_accuracy_score,
)

sns.set_theme(style="whitegrid")
print("Import completati.")

In [ ]:
# ============================================================
# Seed e velocità
# ============================================================
# Fisso il seed su tutte le sorgenti di casualità "controllabili" della
# pipeline: numpy (campionamento stratificato, bootstrap), python random,
# torch (inizializzazione pesi, dropout, split), torch.cuda (kernel GPU).
#
# cudnn.benchmark=True lascia cuDNN libero di provare più algoritmi di
# convoluzione la prima volta che incontra una certa forma di input (qui,
# sempre batch da 32 immagini 224x224) e di tenere il più veloce per tutte
# le iterazioni successive. 
#
# ATTENZIONE!!! La generazione testuale (Qwen con sampling) e la
# generazione di immagini (diffusione) hanno comunque una loro variabilità
# interna gestita con generator dedicati più avanti nel notebook.
# ============================================================
SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# cuDNN è la libreria NVIDIA che PyTorch usa sotto il cofano per eseguire 
# le operazioni pesanti su GPU 
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Device: {device} — dtype usato per i modelli generativi: {dtype}")
if device.type != "cuda":
    print("ATTENZIONE: nessuna GPU rilevata. La generazione delle immagini sintetiche "
          "sarà molto più lenta. Su Colab: Runtime -> Cambia tipo di runtime -> GPU.")

## 3. Configurazione centralizzata

Tutti i parametri del progetto vivono in un'unica dataclass.

In [ ]:
# ============================================================
# Config: parametri centralizzati del progetto
# ============================================================
@dataclass
class Config:
    # --- Dati ---
    # Cartella dove torchvision scarica e mette in cache 
    # Oxford-IIIT Pet (immagini + annotazioni).
    DATA_ROOT: str = "./data"    
    # Cartella dove vengono salvate su disco le immagini 
    # sintetiche generate (organizzate per classe) e il 
    # file synthetic_metadata.csv con caption/prompt/path.
    SYNTH_DIR: str = "./data/synthetic"
    IMG_SIZE: int = 224            # dimensione attesa dal backbone EfficientNet-B0

    # --- Split train/validation sull'intero trainval ---
    TRAIN_SIZE: float = 0.8        # quota (stratificata) del trainval usata per il training; il resto va in validation

    # --- Pipeline di Data Augmentation generativa ---
    AUGMENT_FRACTION: float = 0.3  # quota (stratificata) del training set da aumentare: almeno il 20% richiesto, ne uso il 30%
    N_TEXT_VARIANTS: int = 1       # varianti testuali generate per ogni immagine campionata per l'augmentation
    SYNTH_STEPS: int = 2           # step di diffusione se si usa un modello "turbo" (1-4 step)

    # --- Training del classificatore ---
    BATCH_SIZE: int = 32
    EPOCHS: int = 80
    LR_BACKBONE: float = 1e-5      # LR basso: il backbone e gia pretrained, deve muoversi appena
    LR_HEAD: float = 1e-3          # LR piu alto: la testa parte da pesi casuali, deve imparare in fretta
    WEIGHT_DECAY: float = 5e-4     # alzato da 1e-4: piu regolarizzazione per ridurre l'overfitting osservato
    DROPOUT: float = 0.3           # dropout nella testa di classificazione (via drop_rate di timm)
    LABEL_SMOOTHING: float = 0.1   # smoothing sulle label nella CrossEntropyLoss
    PATIENCE: int = 4               # epoche di pazienza per l'early stopping
    MIN_DELTA: float = 1e-3        # miglioramento minimo di val_loss per non essere considerato rumore

    # --- Modelli pretrained usati (Hugging Face Hub) ---
    # Cinque modelli "dietro le quinte" preparano il materiale di studio (le
    # immagini); solo CLASSIFIER_MODEL è il modello che alleno davvero e
    # di cui misuro le prestazioni.
    CLASSIFIER_MODEL: str = "efficientnet_b0"                     # lo "studente": il classificatore che alleno e valuto
    CAPTION_MODEL: str = "Salesforce/blip-image-captioning-base"  # il "descrittore": guarda una foto vera e la descrive a parole
    TEXT_GEN_MODEL: str = "Qwen/Qwen2.5-1.5B-Instruct"            # lo "scrittore": riscrive la descrizione in modi diversi
    IMAGE_GEN_MODEL: str = "stabilityai/sdxl-turbo"               # il "pittore": disegna una foto finta dalla descrizione
    QUALITY_CHECK_MODEL: str = "openai/clip-vit-base-patch32"     # il "critico d'arte" (modello CLIP): giudica quanto le foto finte assomigliano alla descrizione, non disegna né si allena


cfg = Config()

os.makedirs(cfg.DATA_ROOT, exist_ok=True)
os.makedirs(cfg.SYNTH_DIR, exist_ok=True)
print(cfg)

## 4. Caricamento ed esplorazione di Oxford-IIIT Pet

Il dataset `Oxford-IIIT Pet` contiene immagini di cani e gatti appartenenti a 37 razze ed è già suddiviso in due split:

- trainval → utilizzato per costruire training e validation set;
- test → utilizzato esclusivamente per la valutazione finale.

Viene utilizzato target_types="category" perché il l' obiettivo è classificare la razza dell'animale.

Lo split trainval viene caricato in più versioni perché, nelle diverse fasi del progetto, c'è bisogno di applicare trasformazioni differenti:

- trainval_raw → immagini originali senza trasformazioni, utilizzate anche per il captioning e la generazione dei dati sintetici;
- train_dataset → immagini con le trasformazioni previste per il training;
- val_dataset → immagini con le sole trasformazioni necessarie per la valutazione, senza Data Augmentation.

Anche il test set viene successivamente preparato con le trasformazioni di valutazione, mantenuto separato dai dati di training e validation e utilizzato solo alla fine per valutare e confrontare Baseline e Augmented su immagini mai viste durante il training.

**Nota:** anche se lo stesso split viene caricato più volte, il dataset non viene scaricato nuovamente: torchvision riutilizza i file già presenti nella cartella locale.

In [ ]:
# ============================================================
# Download e caricamento di Oxford-IIIT Pet
# ============================================================
trainval_raw = OxfordIIITPet(
    root=cfg.DATA_ROOT, 
    split="trainval", 
    target_types="category", 
    download=True
)
test_raw = OxfordIIITPet(
    root=cfg.DATA_ROOT, 
    split="test", 
    target_types="category", 
    download=True
)

CLASSES = trainval_raw.classes
NUM_CLASSES = len(CLASSES)

print(f"N. Classi: {NUM_CLASSES} \nCLASSES (razze): {CLASSES}")
print(f"Immagini in trainval: {len(trainval_raw)}")
print(f"Immagini in test: {len(test_raw)}")

In [ ]:
trainval_raw

In [ ]:
import numpy as np
np.unique(trainval_raw._labels)   

In [ ]:
# ============================================================
# Etichette del trainval, per il campionamento stratificato
# ============================================================

# OxfordIIITPet espone internamente la lista delle etichette con "_labels".
if hasattr(trainval_raw, "_labels"):
    trainval_labels = np.array(trainval_raw._labels)
else:
    print("Attributo _labels non trovato: ricostruisco le etichette iterando sul dataset "
          "(più lento, succede solo con versioni di torchvision diverse da quella prevista).")
    trainval_labels = np.array([trainval_raw[i][1] for i in range(len(trainval_raw))])

# Conto quanti elementi appartengono a ciascuna classe
# e ordina il risultato in base all'indice della classe.
class_counts = pd.Series(trainval_labels).value_counts().sort_index()
print(f"Immagini per classe nel trainval — minimo: {class_counts.min()}, "
      f"massimo: {class_counts.max()}, media: {class_counts.mean():.1f}")

In [ ]:
# ============================================================
# Esempi dal dataset
# ============================================================
def view_dataset_example(seed, dataset):
  fig, axes = plt.subplots(2, 6, figsize=(16, 6))
  rng_show = np.random.default_rng(seed)
  sample_idx = rng_show.choice(len(dataset), size=12, replace=False)
  for ax, idx in zip(axes.flat, sample_idx):
      img, label = dataset[idx]
      ax.imshow(img)
      ax.set_title(CLASSES[label].replace("_", " "), fontsize=9)
      ax.axis("off")
  plt.tight_layout()
  plt.show()

In [ ]:
view_dataset_example(SEED, trainval_raw)

### Osservazioni sul dataset

Osservando un primo campione casuale di immagini si nota che il dataset presenta una certa variabilità:

- **Inquadrature diverse:** alcune immagini mostrano il muso in primo piano, altre l'animale intero, di profilo o di spalle.
- **Sfondi diversi:** gli animali possono trovarsi in casa, all'aperto o davanti a sfondi neutri.
- **Illuminazione variabile:** luminosità e presenza di ombre cambiano tra le immagini.
- **Pose diverse:** gli animali possono trovarsi in posizioni e orientamenti differenti.
- **Razze simili:** alcune classi sono visivamente molto simili e quindi possono essere più difficili da distinguere.
- **Numero limitato di immagini:** il modello potrebbe imparare caratteristiche poco utili, come lo sfondo, invece di concentrarsi sull'animale.

Questa variabilità rende il problema di classificazione più complesso, ma è anche simile a quello che potrebbe succedere in un caso reale, ad esempio con immagini provenienti da telecamere di sorveglianza, dove inquadratura, luce e qualità dell'immagine possono cambiare.

→ Per questo motivo utilizzerò la **Data Augmentation**, in modo da aumentare la varietà delle immagini viste durante il training e aiutare il modello a riconoscere gli animali anche in condizioni diverse.

In [ ]:
# ============================================================
# Distribuzione delle classi nel trainval
# ============================================================
def class_distribution(num_classes, class_counts, name_dataset,
                        class_counts_2=None, label_1=None, label_2=None, title=None):
    plt.figure(figsize=(14, 5))
    if class_counts_2 is None:
        # Caso base: una sola serie (usato per trainval/train/val).
        plt.bar(range(num_classes), class_counts.values, color="#4C72B0")
        plt.ylabel(f"Numero di immagini nel {name_dataset}")
        plt.title(title or f"Distribuzione delle immagini per razza — Oxford-IIIT Pet ({name_dataset})")
    else:
        # Caso a due serie affiancate (non sovrapposte): usato per
        # confrontare due composizioni diverse dello stesso insieme di
        # classi (es. trainval completo vs. training aumentato). Le barre
        # sono spostate di +-width/2 rispetto all'indice della classe,
        # altrimenti una coprirebbe completamente l'altra.
        width = 0.4
        x = np.arange(num_classes)
        plt.bar(x - width / 2, class_counts.reindex(range(num_classes), fill_value=0).values,
                width=width, color="#4C72B0", label=label_1)
        plt.bar(x + width / 2, class_counts_2.reindex(range(num_classes), fill_value=0).values,
                width=width, color="#DD8452", label=label_2)
        plt.ylabel("Numero di immagini")
        plt.title(title)
        plt.legend()
    plt.xlabel("Indice classe (razza)")
    plt.tight_layout()
    plt.show()

In [ ]:
class_distribution(NUM_CLASSES, class_counts, "trainval")

Il dataset trainval risulta già abbastanza bilanciato: le 37 razze hanno un numero di immagini simile, generalmente compreso tra circa 90 e 100 campioni per classe.

Uso l'intero trainval per il training e la validation, suddividendolo in modo percentuale e stratificato con `train_test_split` di scikit-learn: nessun sotto-campionamento artificiale, ogni razza contribuisce con tutte le immagini disponibili nella proporzione scelta.

## 5. Costruzione degli split: training, validation, test

Costruisco tre insiemi **disgiunti** a partire dall'intero trainval:

- **Training** (`TRAIN_SIZE` = 80% del trainval, campionato in modo stratificato con `train_test_split`): usato per allenare entrambi i classificatori, con e senza augmentation.
- **Validation** (il restante 20% del trainval stratificato): serve per l'early stopping e per scegliere quando fermare il training. Rimane sempre composto da sole immagini reali, in nessuno dei due esperimenti viene "aumentato".
- **Test** (lo split ufficiale `test` di Oxford-IIIT Pet): usato una sola volta per ciascun modello.

Per la pipeline di Data Augmentation generativa non viene usato tutto il training set: viene campionato un ulteriore 30% stratificato del training (`AUGMENT_FRACTION`) da cui generare le immagini sintetiche.

In [ ]:
# ============================================================
# Split train/validation con scikit-learn
# ============================================================
# Oxford-IIIT Pet fornisce già un test set separato, che quindi qui
# non tocco: divido solo lo split "trainval" in training e validation.
# ============================================================

# prendo tutti gli indici di trainval, creando un range
all_indices = np.arange(len(trainval_labels))

# train_test_split(*arrays ...) > passo l'array di indici
train_idx, val_idx = train_test_split(
    all_indices,
    train_size=cfg.TRAIN_SIZE,   # dimensione di train (80%)
    stratify=trainval_labels,    # mantieni tra train e val stessa distribuzione di ogni razza
    random_state=SEED,
)
train_idx, val_idx = sorted(train_idx.tolist()), sorted(val_idx.tolist())

In [ ]:
print("Train split: ", train_idx)
print("Val split: ", val_idx)

In [ ]:
print(f"Training: {len(train_idx)} immagini "
      f"({len(train_idx) / NUM_CLASSES:.1f} per classe in media)")
print(f"Validation: {len(val_idx)} immagini "
      f"({len(val_idx) / NUM_CLASSES:.1f} per classe in media)")
print(f"Test (ufficiale, mai campionato): {len(test_raw)} immagini")

overlap = set(train_idx) & set(val_idx)
assert len(overlap) == 0, "Training e validation si sovrappongono: controllare lo split."
print("Nessuna sovrapposizione tra training e validation: verificato.")

## 6. Trasformazioni e DataLoader

Vengono settate le stesse trasformazioni **per la baseline e per l' augmented**.

- **Training**: crop casuale ridimensionato, flip orizzontale e piccola rotazione, poi resize a 224×224 e normalizzazione con le statistiche di ImageNet (il backbone è pretrained su ImageNet: usare le stesse statistiche di normalizzazione dell'addestramento originale evita di spostare la distribuzione dei pixel fuori dal range che il modello si aspetta).
- **Validation/Test**: solo resize e normalizzazione, senza augmentation — le metriche di valutazione devono riflettere immagini "reali", non trasformate.

In [ ]:
# ============================================================
# Trasformazioni per training e valutazione
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(
        cfg.IMG_SIZE,
        scale=(0.65, 1.0)  # range allargato da (0.8, 1.0): piu variabilita' nell'inquadratura
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(cfg.IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Creo i dataset completi per trai_dataset e val_dataset in modo 
# da poterne poi estrarre i sottoinsiemi train/val.
train_tmp = OxfordIIITPet(
    root=cfg.DATA_ROOT, 
    split="trainval",
    target_types="category", 
    transform=train_transform
)
val_tmp = OxfordIIITPet(
    root=cfg.DATA_ROOT, 
    split="trainval",
    target_types="category", 
    transform=eval_transform
)
test_dataset = OxfordIIITPet(
    root=cfg.DATA_ROOT, 
    split="test",
    target_types="category", 
    transform=eval_transform
)

# Subset prende il dataset completo (trainval) e la lista di indici 
# (train_idx, val_idx), e crea una vista ristretta per train e val dataset
train_dataset = Subset(train_tmp, train_idx)
val_dataset = Subset(val_tmp, val_idx)

print(f"train_dataset: {len(train_dataset)} immagini")
print(f"val_dataset: {len(val_dataset)} immagini")
print(f"test_dataset: {len(test_dataset)} immagini")

In [ ]:
# Recupero le label SOLO delle immagini appartenenti al train_dataset
train_labels_dataset = np.array([
    train_tmp[i][1] for i in train_idx
])

# Conto quanti elementi appartengono a ciascuna classe
# e ordina il risultato in base all'indice della classe.
class_counts_train_dataset = pd.Series(train_labels_dataset).value_counts().sort_index()

class_distribution(NUM_CLASSES, class_counts_train_dataset, "train")

In [ ]:
# Recupero le label SOLO delle immagini appartenenti al val_dataset
val_labels_dataset = np.array([
    val_tmp[i][1] for i in val_idx
])

# Conto quanti elementi appartengono a ciascuna classe
# e ordina il risultato in base all'indice della classe.
class_counts_dataset_val = pd.Series(val_labels_dataset).value_counts().sort_index()

class_distribution(NUM_CLASSES, class_counts_dataset_val, "val")

### Verifica della distribuzione dopo lo split

Dopo la suddivisione del dataset trainval in training set e validation set, la distribuzione delle 37 classi rimane sostanzialmente bilanciata in entrambi i sottoinsiemi. Questo garantisce che tutte le razze continuino a essere adeguatamente rappresentate sia durante l'addestramento sia durante la validazione del modello.

In [ ]:
# ============================================================
# DataLoader condivisi
# ============================================================
# DEBUG TEMPORANEO: num_workers=0 per capire se e' val_loader la causa del
# crash "DataLoader worker exited unexpectedly" visto durante un training
# precedente. Rimettere num_workers=2, persistent_workers=True se il problema
# non si ripresenta (richiede di rieseguire questa cella per ricreare val_loader).
val_loader = DataLoader(
    val_dataset, 
    batch_size=cfg.BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=cfg.BATCH_SIZE, 
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

baseline_train_loader = DataLoader(
    train_dataset, 
    batch_size=cfg.BATCH_SIZE,
    shuffle=True, 
    num_workers=2,
    persistent_workers=True
)

## 7. Architettura del classificatore e classi di supporto

### Classificatore: EfficientNet-B0

Per entrambi gli esperimenti viene utilizzato EfficientNet-B0 pre-addestrato, creato tramite: `timm.create_model("efficientnet_b0", pretrained=True)`, con un dropout (`DROPOUT`) inserito prima della testa di classificazione e un weight decay (`WEIGHT_DECAY`) applicato dall'ottimizzatore, per contenere l'overfitting osservato nei primi esperimenti (il percorso di tentativi che mi ha portato a questi valori non viene riportato per esteso, altrimenti il notebook diventerebbe troppo lungo). 
Viene utilizzato `timm` invece di `torchvision.models` perché funziona sempre allo stesso modo, qualunque modello si scelga: per provare un'altra architettura basta cambiare `CLASSIFIER_MODEL`, senza toccare il resto del codice. Con `torchvision`, invece, ogni modello ha un nome diverso per la testa da sostituire (`.fc`, `.classifier`, `.head`...), quindi cambiare architettura richiederebbe più lavoro.

Il modello parte quindi dalle caratteristiche visive già imparate durante il pre-training. La testa finale viene sostituita con una nuova, adatta a riconoscere le 37 razze di Oxford-IIIT Pet. Il backbone non viene bloccato: si allena tutto il modello, ma con due velocità diverse -- `LR_BACKBONE` basso per il backbone, così non "dimentica" troppo in fretta quello che ha già imparato, e `LR_HEAD` più alto per la testa, che invece deve imparare tutto da zero. In più, la loss usa un po' di label smoothing (`LABEL_SMOOTHING`): serve a evitare che il modello diventi troppo sicuro delle proprie previsioni sul training set.

## Classi e funzioni di supporto

- **`Trainer`** → incapsula il ciclo di training/validazione di un'epoca ed effettua l'early stopping sulla validation loss, salvando i pesi del checkpoint migliore.
- **`Evaluator`** → valuta il modello calcolando le diverse metriche su un DataLoader (accuracy, top-3 accuracy, precision, recall e F1-score) salvando le info necessarie per produrre grafici come la matrice di confusione.
- **`build_model()`** → crea il modello EfficientNet-B0 con la configurazione scelta. Viene richiamata separatamente per Baseline e Augmented, così entrambi partono dalla stessa architettura e configurazione iniziale.

In questo modo training, valutazione e architettura rimangono gli stessi nei due esperimenti: ciò che cambia è il training set, che nel secondo caso viene arricchito con le immagini sintetiche.

In [ ]:
# ============================================================
# build_model: crea un classificatore EfficientNet-B0 con testa nuova
# Creazione di EfficientNet-B0 pre-addestrata e adattata la testa finale
# alle 37 classi del dataset Oxford-IIIT Pet.
# ============================================================

def build_model():
    """
    Costruisce il modello di classificazione.

    STRUTTURA CONCETTUALE:

        immagine
           ↓
        BACKBONE
        EfficientNet-B0 pretrained
           ↓
        feature visive
           ↓
        NUOVA TESTA DI CLASSIFICAZIONE
        adattata alle 37 classi
           ↓
        previsione della classe

    BACKBONE:
    È la parte principale di EfficientNet-B0 che estrae dalle immagini
    caratteristiche visive utili, come bordi, texture, forme e pattern
    progressivamente più complessi.
    PRETRAINED:
    Il backbone viene inizializzato con pesi già appresi su ImageNet.
    In questo modo il modello non parte completamente da zero, ma dispone
    già di feature visive generali utili.
    NUOVA TESTA:
    La testa di classificazione originale di EfficientNet-B0 viene sostituita
    con una nuova testa avente NUM_CLASSES output, cioè 37 nel nostro caso.

    PERCHE' TIMM (invece di torchvision.models):
    num_classes sostituisce automaticamente la testa finale con un'interfaccia
    identica per qualunque backbone della libreria -- con torchvision andrebbe
    fatto a mano, e il nome dell'attributo da sostituire cambia da
    un'architettura all'altra. Allo stesso modo drop_rate espone il dropout
    come parametro pronto all'uso (vedi sotto). Grazie a questo, provare un
    backbone diverso (uno degli sviluppi futuri elencati nelle conclusioni)
    richiede solo di cambiare CLASSIFIER_MODEL in Config, senza toccare
    questa funzione.

    FINE-TUNING:
    Il backbone NON viene congelato.
    Tutti i suoi parametri rimangono allenabili e possono quindi adattarsi
    al nuovo problema di classificazione.

    In seguito, nel Trainer, verranno utilizzati due learning rate:
    - più basso per il backbone, per modificare gradualmente le feature pretrained;
    - più alto per la nuova testa, che deve imparare il nuovo task.

    REGOLARIZZAZIONE:
    drop_rate=cfg.DROPOUT inserisce un dropout prima della testa finale
    (parametro nativo di timm per le EfficientNet), per ridurre l'overfitting
    osservato nei primi esperimenti.
    """
    model = timm.create_model(
        cfg.CLASSIFIER_MODEL, 
        pretrained=True, 
        num_classes=NUM_CLASSES,
        drop_rate=cfg.DROPOUT,
      )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Parametri allenabili: {trainable:,} / {total:,} "
          f"({100 * trainable / total:.2f}% del totale)")
    return model.to(device)

In [ ]:
# ============================================================
# Trainer: ciclo di training con early stopping
# ============================================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, epochs=cfg.EPOCHS,
                 lr_backbone=cfg.LR_BACKBONE, lr_head=cfg.LR_HEAD,
                 patience=cfg.PATIENCE, min_delta=cfg.MIN_DELTA, label=""):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.patience = patience
        self.min_delta = min_delta
        self.label = label
        self.criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)

        # Learning rate differenziati: il backbone e gia pretrained e deve
        # muoversi appena per non "dimenticare" le feature imparate su
        # ImageNet, mentre la testa parte da pesi casuali e deve imparare
        # molto piu in fretta. 
        # AdamW (invece di Adam) applica il weight decay in modo 
        # disaccoppiato dal learning rate adattivo.
        backbone_params = [p for n, p in model.named_parameters() if "classifier" not in n]
        head_params = [p for n, p in model.named_parameters() if "classifier" in n]
        self.optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params, "lr": lr_head},
        ], weight_decay=cfg.WEIGHT_DECAY)

        self.history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    def _run_epoch(self, loader, train):
        self.model.train() if train else self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.set_grad_enabled(train):
            for images, labels in loader:
                images, labels = images.to(device), labels.to(device)
                if train:
                    self.optimizer.zero_grad()
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                if train:
                    loss.backward()
                    self.optimizer.step()
                total_loss += loss.item() * images.size(0)
                correct += (outputs.argmax(dim=1) == labels).sum().item()
                total += images.size(0)
        return total_loss / total, correct / total

    def fit(self):
        """
        Esegue l'intero processo di addestramento del modello.
        Per ogni epoca:
        1. esegue una fase di TRAINING tramite _run_epoch(..., train=True);
        2. esegue una fase di VALIDATION tramite _run_epoch(..., train=False);
        3. salva loss e accuracy di training e validation in self.history;
        4. controlla se la validation loss è migliorata di almeno 1e-4;
        5. se è migliorata, salva una copia dei pesi del modello;
        6. se non migliora per un certo numero di epoche consecutive
          (self.patience), interrompe anticipatamente il training
          tramite EARLY STOPPING.          
        Al termine, ripristina i pesi del modello relativi alla migliore
        validation loss osservata durante il training.

        INPUT:
        - Utilizza gli attributi dell'oggetto, tra cui:
            self.model        -> modello da addestrare
            self.train_loader -> dati di training
            self.val_loader   -> dati di validation
            self.epochs       -> numero massimo di epoche
            self.patience     -> numero di epoche consecutive senza un
                                miglioramento sufficiente della validation loss
                                tollerate prima dell'early stopping
            self.history      -> dizionario che salva:
                                train_loss, val_loss, train_acc, val_acc
        OUTPUT:
        - self.model   -> modello con i pesi della migliore epoca
        - self.history -> andamento di loss e accuracy durante il training
        """
        # Inizialmente viene impostata a infinito, così la loss della
        # prima epoca sarà sicuramente considerata un miglioramento.
        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0
        for epoch in range(1, self.epochs + 1):
            # TRAINING
            train_loss, train_acc = self._run_epoch(self.train_loader, train=True)
            # VALIDATION
            val_loss, val_acc = self._run_epoch(self.val_loader, train=False)
            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_acc"].append(val_acc)
            print(f"[{self.label}] Epoca {epoch:02d}/{self.epochs} - "
                  f"train_loss {train_loss:.4f} acc {train_acc:.3f} | "
                  f"val_loss {val_loss:.4f} acc {val_acc:.3f}")
            if val_loss < best_val_loss - self.min_delta:
                best_val_loss = val_loss
                best_state = copy.deepcopy(self.model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print(f"[{self.label}] Early stopping all'epoca {epoch} "
                          f"(nessun miglioramento della val_loss da {self.patience} epoche).")
                    break
        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self.model, self.history

In [ ]:
# ============================================================
# Evaluator: metriche e grafici, riusati identici per i due esperimenti
# ============================================================
class Evaluator:
    def __init__(self, model, class_names):
        self.model = model
        self.class_names = class_names

    @torch.no_grad()
    def predict(self, loader):
        self.model.eval()
        all_labels, all_probs = [], []
        for images, labels in loader:
            images = images.to(device)
            logits = self.model(images)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
        return np.concatenate(all_labels), np.concatenate(all_probs)

    def evaluate(self, loader):
        """
        Valuta le prestazioni del modello sui dati contenuti nel DataLoader ricevuto.
        La funzione:
        1. ottiene, tramite predict(), le etichette reali e le probabilità
          previste dal modello;
        2. trasforma le probabilità nelle classi predette;
        3. calcola diverse metriche di classificazione:
          - Accuracy
          - Top-3 Accuracy
          - Classification Report
          - Confusion Matrix

        INPUT:
        - loader:
            DataLoader contenente le immagini da valutare e le rispettive label.
            Può essere, ad esempio, il validation loader o il test loader.

        OUTPUT:
        - restituisce un dizionario contenente:
            labels              -> classi reali
            probs               -> probabilità predette dal modello
            preds               -> classi predette
            accuracy            -> accuracy globale
            top3_accuracy       -> Top-3 accuracy
            precision_macro     -> precision media dando lo stesso peso a ogni classe
            recall_macro        -> recall medio dando lo stesso peso a ogni classe
            f1_macro            -> F1-score medio dando lo stesso peso a ogni classe
            precision_weighted  -> precision pesata per il numero di esempi di ogni classe
            recall_weighted     -> recall pesato per il numero di esempi di ogni classe
            f1_weighted         -> F1-score pesato per il numero di esempi di ogni classe
            report              -> metriche dettagliate per ogni singola classe
            confusion_matrix    -> matrice degli errori tra classi reali e predette
        """
        labels, probs = self.predict(loader)
        preds = probs.argmax(axis=1)
        accuracy = accuracy_score(labels, preds)
        top3_accuracy = top_k_accuracy_score(labels, probs, k=3, labels=range(len(self.class_names)))
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            labels, preds, average="weighted", zero_division=0
        )
        report = classification_report(
            labels, preds, target_names=self.class_names, zero_division=0, output_dict=True
        )
        cm = confusion_matrix(labels, preds)
        return {
            "labels": labels, 
            "probs": probs, 
            "preds": preds,
            "accuracy": accuracy, 
            "top3_accuracy": top3_accuracy,
            "precision_macro": precision_macro, 
            "recall_macro": recall_macro, 
            "f1_macro": f1_macro,
            "precision_weighted": precision_w, 
            "recall_weighted": recall_w, 
            "f1_weighted": f1_w,
            "report": report, 
            "confusion_matrix": cm,
        }

    def plot_confusion_matrix(self, cm, title):
        plt.figure(figsize=(14, 12))
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        sns.heatmap(cm_norm, cmap="Blues", xticklabels=self.class_names,
                    yticklabels=self.class_names, cbar_kws={"label": "quota sul totale della classe reale"})
        plt.xlabel("Classe predetta")
        plt.ylabel("Classe reale")
        plt.title(title)
        plt.xticks(rotation=90, fontsize=6)
        plt.yticks(rotation=0, fontsize=6)
        plt.tight_layout()
        plt.show()

    def plot_history(self, history, title):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        axes[0].plot(history["train_loss"], label="train")
        axes[0].plot(history["val_loss"], label="validation")
        axes[0].set_title(f"{title} — loss")
        axes[0].set_xlabel("epoca")
        axes[0].legend()
        axes[1].plot(history["train_acc"], label="train")
        axes[1].plot(history["val_acc"], label="validation")
        axes[1].set_title(f"{title} — accuracy")
        axes[1].set_xlabel("epoca")
        axes[1].legend()
        plt.tight_layout()
        plt.show()

    def hardest_classes(self, results, top_n=10):
        """
        Individua le classi più difficili da riconoscere.
        Per ogni classe considera il RECALL, cioè:
            esempi correttamente classificati della classe
            ------------------------------------------------
            numero totale di esempi reali appartenenti alla classe
        Un recall basso indica che il modello ha difficoltà
        a riconoscere correttamente quella classe.

        INPUT:
        - results: dizionario restituito da evaluate()
        - top_n: numero di classi più difficili da mostrare
        OUTPUT:
        - DataFrame ordinato dalla classe con recall più basso
        a quella con recall più alto.
        """

        report = results["report"]
        rows = []
        for class_name in self.class_names:
            rows.append({
                "Classe": class_name,
                "Precision": report[class_name]["precision"],
                "Recall": report[class_name]["recall"],
                "F1-score": report[class_name]["f1-score"],
                "Support": report[class_name]["support"]
            })

        df = pd.DataFrame(rows)
        # Ordina dalla classe con recall più basso,
        # cioè quella che il modello riconosce peggio.
        df = df.sort_values("Recall", ascending=True)
        return df.head(top_n)

    def most_confused_classes(self, results, top_n=15):
        """
        Individua le confusioni più frequenti tra classi.
        Analizza la confusion matrix e considera solamente
        le celle FUORI dalla diagonale.
        Esempio:
            Classe reale: American Pit Bull Terrier
            Predetta come: Staffordshire Bull Terrier
            Errori: 12
        significa che 12 immagini appartenenti realmente alla prima
        classe sono state classificate come appartenenti alla seconda.

        INPUT:
        - results: dizionario restituito da evaluate()
        - top_n: numero massimo di confusioni da mostrare
        OUTPUT:
        - DataFrame ordinato dalle confusioni più frequenti
        a quelle meno frequenti.
        """

        cm = results["confusion_matrix"]
        rows = []
        for i in range(len(self.class_names)):
            for j in range(len(self.class_names)):
                # i == j rappresenta una classificazione corretta.
                # A noi interessano solo gli errori.
                if i != j and cm[i, j] > 0:
                    rows.append({
                        "Classe reale": self.class_names[i],
                        "Predetta come": self.class_names[j],
                        "Numero errori": cm[i, j]
                    })

        df = pd.DataFrame(rows)
        df = df.sort_values(
            "Numero errori",
            ascending=False
        )
        return df.head(top_n)

## 8. Esperimento A — Baseline (solo dati reali)

Alleno il primo classificatore usando **solo** `train_dataset`: le immagini reali dell'80% del trainval, senza alcuna aggiunta sintetica. Rappresenta il sistema di riconoscimento immagini che CyberEye ha oggi, prima di qualunque intervento di Data Augmentation. Il validation set reale serve per l'early stopping.

In [ ]:
# ============================================================
# Training del modello Baseline
# ============================================================
baseline_model = build_model()
baseline_trainer = Trainer(
    baseline_model, 
    baseline_train_loader, 
    val_loader,
    epochs=cfg.EPOCHS, 
    lr_backbone=cfg.LR_BACKBONE, 
    lr_head=cfg.LR_HEAD,
    patience=cfg.PATIENCE, 
    min_delta=cfg.MIN_DELTA,
    label="baseline",
)
baseline_model, baseline_history = baseline_trainer.fit()

In [ ]:
# ============================================================
# Curve di training del modello Baseline
# ============================================================
evaluator = Evaluator(baseline_model, CLASSES)
evaluator.plot_history(baseline_history, "Esperimento Baseline")

### Analisi delle curve di training — Baseline

Dalle curve si vede che il modello **impara molto velocemente nelle prime epoche**. La training loss scende da circa **3.11 a 1.21 già nelle prime 6 epoche**, mentre la training accuracy passa dal **24,6% a circa l'85%**.

Continuando il training, il modello migliora sempre di più sui dati di training: nelle ultime epoche la **training accuracy arriva quasi al 100% (fino al 99,8% nell'ultima epoca)**, mentre la training loss continua gradualmente a diminuire fino a circa **0,79**.

Anche sul validation set il modello migliora molto all'inizio. La **validation accuracy passa dal 61,3% della prima epoca a circa l'86-87%** già entro le prime 10 epoche, poi continua a crescere più lentamente, oscillando nelle epoche finali tra l'**89% e il 91%**, senza un andamento perfettamente monotono.

La validation loss continua a diminuire più lentamente rispetto alla training loss, scendendo sotto l'1.00 dall'epoca 36 circa, fino a raggiungere il valore migliore intorno a **0,989, tra le epoche 43 e 46**. Nelle epoche successive non si osservano ulteriori miglioramenti sufficienti e la loss rimane intorno a **0,99-1,00**.

Si nota quindi una distanza crescente tra training e validation: il modello arriva a classificare quasi perfettamente i dati di training, mentre sulla validation le prestazioni si stabilizzano intorno al 90%. Questo suggerisce la presenza di **overfitting**, anche se il modello mantiene comunque buone prestazioni sui dati di validation.

L'**early stopping** interrompe il training all'epoca **47**, dopo 4 epoche senza un nuovo miglioramento sufficiente della validation loss. In questo modo non è necessario arrivare alle 80 epoche massime previste.

> **Nota:** in questo esperimento viene utilizzato il **label smoothing**, quindi il valore della loss non deve necessariamente avvicinarsi a zero anche quando l'accuracy è molto alta. Per questo motivo una training loss intorno a **0,79-0,80** è compatibile con una training accuracy vicina al **99,8%**.

→ Nel complesso, il modello Baseline ha raggiunto una buona capacità di classificazione, ma il divario tra training e validation mostra che nelle ultime epoche continua soprattutto a migliorare sui dati già visti, mentre la capacità di generalizzare tende a stabilizzarsi.

## 9. Valutazione dell'esperimento Baseline sul test set

Prima valutazione sul test set ufficiale, mai visto finora. La ripeto in modo identico più avanti per l'esperimento augmented, cosi il confronto finale è diretto.

In [ ]:
# ============================================================
# Metriche Baseline sul test set
# ============================================================
baseline_results = evaluator.evaluate(test_loader)

print(f"Accuracy:            {baseline_results['accuracy']:.4f}")
print(f"Top-3 accuracy:       {baseline_results['top3_accuracy']:.4f}")
print(f"Precision (macro):   {baseline_results['precision_macro']:.4f}")
print(f"Recall (macro):      {baseline_results['recall_macro']:.4f}")
print(f"F1 (macro):          {baseline_results['f1_macro']:.4f}")
print(f"F1 (weighted):       {baseline_results['f1_weighted']:.4f}")
print(baseline_results['report'])

In [ ]:
# ============================================================
# Matrice di confusione Baseline
# ============================================================
evaluator.plot_confusion_matrix(
    baseline_results["confusion_matrix"], "Matrice di confusione — Baseline (test set)"
)

In [ ]:
# ============================================================
# Classi più difficili per il modello Baseline
# ============================================================

hardest_baseline = evaluator.hardest_classes(
    baseline_results,
    top_n=10
)

hardest_baseline

In [ ]:
# ============================================================
# Confusioni più frequenti del modello Baseline
# ============================================================

confusions_baseline = evaluator.most_confused_classes(
    baseline_results,
    top_n=15
)

confusions_baseline

### Valutazione del modello Baseline sul test set

Sul test set il modello Baseline raggiunge una **Accuracy dell'88,66%**, quindi riesce a classificare correttamente come prima scelta quasi **89 immagini su 100**.

La **Top-3 Accuracy è invece del 97,11%**. Questo significa che, anche quando la prima classe scelta dal modello non è quella corretta, nella maggior parte dei casi la classe reale si trova comunque tra le **3 classi considerate più probabili**.

Le altre metriche ottenute sono:

- **Precision Macro: 88,96%**
- **Recall Macro: 88,61%**
- **F1 Macro: 88,45%**
- **F1 Weighted: 88,52%**

Precision, Recall e F1 hanno valori abbastanza vicini tra loro, quindi non emerge una grande differenza tra queste misure a livello complessivo.

Anche **F1 Macro e F1 Weighted sono molto simili**. Questo è coerente con il fatto che le classi del test set hanno un numero di esempi abbastanza simile e quindi dare lo stesso peso ad ogni classe oppure pesarlo in base alla numerosità non cambia molto il risultato finale.

### Matrice di confusione e classi più difficili

Osservando la matrice di confusione si nota una **diagonale molto evidente**. Questo significa che, per la maggior parte delle 37 razze, molte immagini vengono classificate correttamente.

Guardando però le metriche delle singole classi, alcune razze risultano più difficili di altre.

La classe più problematica è **American Pit Bull Terrier**, che presenta:

- **Precision: 80,77%**
- **Recall: 42,00%**
- **F1-score: 55,26%**

Il recall del 42% indica che il modello riesce a riconoscere correttamente poco più di **4 immagini su 10** appartenenti a questa razza.

Tra le altre classi più difficili troviamo:

- **Staffordshire Bull Terrier:** precision 60,00%, recall ≈ **67,4%**, F1 ≈ **63,5%**
- **Ragdoll:** recall **69%**, F1 ≈ **71,5%**
- **Maine Coon:** recall **77%**, F1 ≈ **82,4%**
- **British Shorthair:** recall **78%**, F1 ≈ **83,4%**
- **Egyptian Mau:** recall **80,4%**, F1 ≈ **86,2%**

Quindi le prestazioni complessive sono buone, ma non sono uguali per tutte le razze.

### Confusioni più frequenti

Per capire meglio gli errori è utile osservare **con quale razza viene confusa una classe quando il modello sbaglia**.

La confusione più frequente è:

- **American Pit Bull Terrier → Staffordshire Bull Terrier: 25 errori**

Seguono:

- **Ragdoll → Birman: 16 errori**
- **Egyptian Mau → Bengal: 15 errori**
- **Staffordshire Bull Terrier → American Bulldog: 12 errori**
- **American Pit Bull Terrier → American Bulldog: 12 errori**
- **Birman → Ragdoll: 10 errori**
- **Siamese → Birman: 10 errori**
- **Basset Hound → Beagle: 9 errori**
- **British Shorthair → Russian Blue: 9 errori**
- **Maine Coon → Bengal: 9 errori**

Alcune confusioni sembrano avvenire tra razze che possono avere **caratteristiche visive simili**. Un esempio evidente è `American Pit Bull Terrier`, che viene confuso soprattutto con `Staffordshire Bull Terrier` e `American Bulldog`.

Si nota anche che alcune confusioni avvengono in entrambe le direzioni, ad esempio:

- `Ragdoll → Birman` (16 errori)
- `Birman → Ragdoll` (10 errori)

oppure, con numeri più piccoli:

- `Egyptian Mau → Bengal` (15 errori)
- `Bengal → Egyptian Mau` (6 errori)

Questo fa capire che il problema è di tipo **fine-grained**: il modello non deve semplicemente distinguere un cane da un gatto, ma deve riconoscere razze anche molto simili tra loro.

La **Top-3 Accuracy del 97,11%** è interessante proprio in questo contesto: anche se l'Accuracy è dell'88,66%, molto spesso la classe corretta compare comunque tra le prime tre alternative del modello.

### Considerazioni sul Baseline

Nel complesso, il modello Baseline raggiunge **buone prestazioni sul test set**, con un'Accuracy vicina all'89% e una Top-3 Accuracy superiore al 97%.

L'analisi per classe mostra però che il risultato globale può nascondere alcune difficoltà specifiche. In particolare, **American Pit Bull Terrier** e **Staffordshire Bull Terrier** risultano tra le classi più problematiche e vengono anche confuse frequentemente tra loro.

→ Questi risultati rappresentano il **punto di riferimento del Baseline**. Nel confronto con il modello addestrato utilizzando anche le immagini sintetiche sarà quindi utile controllare non solo se migliorano Accuracy e F1 complessivi, ma anche se diminuiscono gli errori sulle classi più difficili e sulle coppie di razze che il Baseline tende a confondere.

In [ ]:
def clear_gpu_memory():
    """
    Libera la memoria non più utilizzata.
    - gc.collect() forza il garbage collector di Python a eliminare
      gli oggetti non più referenziati.
    - torch.cuda.empty_cache() libera la memoria GPU mantenuta nella
      cache di PyTorch, rendendola disponibile per utilizzi successivi.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(
            f"Memoria GPU attualmente allocata: "
            f"{torch.cuda.memory_allocated() / 1e9:.2f} GB"
        )

In [ ]:
# ============================================================
# Classi per la pipeline generativa: carica -> genera -> libera
# Una classe per fase (captioning, generazione testuale, generazione
# immagini), ciascuna con lo stesso schema load_model() -> genera -> clear_gpu().
# ============================================================
class Blip:
    def __init__(self, model_name):
        self.model_name = model_name
        self.processor = None
        self.model = None

    def load_model(self):
        # processor: immagine -> numeri (traduttore) > MULTIMODALE
        self.processor = BlipProcessor.from_pretrained(self.model_name)
        # model: numeri -> nuovi token (la generazione vera)
        self.model = BlipForConditionalGeneration.from_pretrained(self.model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def caption(self, image):
        """
        Genera una descrizione testuale (caption) di una singola immagine
        utilizzando il modello BLIP pre-addestrato.

        INPUT:
        - image: immagine da descrivere.
        OUTPUT:
        - stringa contenente la caption generata da BLIP.

        La funzione:
        1. prepara l'immagine tramite BlipProcessor;
        2. esegue BLIP in modalità inference;
        3. genera al massimo 30 nuovi token;
        4. converte i token generati in testo leggibile.
        """
        inputs = self.processor(image, return_tensors="pt").to(device)
        output_ids = self.model.generate(**inputs, max_new_tokens=30)
        # processor: token -> testo leggibile
        return self.processor.decode(output_ids[0], skip_special_tokens=True)

    def run_captioning(self, indices):
        """
        Captioning di tutte le immagini campionate per l'augmentation
        Genera una caption per ogni immagine presente nel sottoinsieme campionato.
        Per ogni immagine:
        1. recupera immagine e label dal dataset originale;
        2. genera una descrizione testuale tramite BLIP;
        3. salva in un dizionario:
          - indice originale dell'immagine
          - label numerica
          - nome della razza
          - caption generata
        """
        records = []
        for idx in tqdm(indices, desc="Captioning con BLIP"):
            image, label = trainval_raw[idx]
            # Creo un dizionario con tutte le informazioni utili
            # associate all'immagine corrente e lo aggiunge alla lista.
            records.append({
                "source_idx": idx,
                "label": label,
                "breed": CLASSES[label],
                "breed_name": CLASSES[label].replace("_", " "),
                "caption": self.caption(image),
            })
        return records

    def clear_gpu(self):
        """
        Liberazione di BLIP dalla memoria GPU
        """
        del self.model, self.processor
        self.model, self.processor = None, None
        clear_gpu_memory()

In [ ]:
class Qwen:
    def __init__(self, model_name):
        self.model_name = model_name
        self.tokenizer = None
        self.model = None

    def load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name, torch_dtype=dtype).to(device)
        self.model.eval()

    @torch.no_grad()
    def generate_variants(self, caption, n_variants):
        """
        Genera n_variants parafrasi della sola parte descrittiva
        della caption originale.

        INPUT:
        - caption: stringa contenente la caption prodotta da BLIP.
        - n_variants: numero di varianti testuali da generare.
        OUTPUT:
        - lista di stringhe contenente le nuove descrizioni generate.

        Importante:
        la funzione NON riceve il nome della razza.
        In questo modo Qwen non può modificarlo accidentalmente.
        """
        messages = [
            {"role": "system", "content": "You rewrite short image descriptions. "
                                           "Reply with only the rewritten sentence, nothing else."},
            {"role": "user", "content": f"Rewrite this image description in a different way, "
                                         f"keeping the same meaning, in one short sentence: {caption}"},
        ]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=True,
            top_k=50,
            temperature=0.9,
            num_return_sequences=n_variants,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
        return [self.tokenizer.decode(t, skip_special_tokens=True).strip() for t in new_tokens]

    def run_variant_generation(self, caption_records, n_variants):
        """
        Generazione delle varianti testuali per tutte le caption
        Vengono generate le varianti testuali di tutte le caption prodotte 
        da BLIP e costruisce i prompt finali che verranno poi utilizzati
        per generare le immagini sintetiche.
        Per ogni record presente in caption_records:
        1. prende la caption generata da BLIP;
        2. genera più varianti testuali tramite Qwen;
        3. associa ogni variante alla label e alla razza originali;
        4. costruisce il prompt finale reinserendo la razza reale
           davanti alla descrizione generata;
        5. salva tutte le informazioni in synth_prompts.
        """
        prompts = []
        for rec in tqdm(caption_records, desc="Generazione varianti con Qwen"):
            # genero N_TEXT_VARIANTS caption variabili
            variants = self.generate_variants(rec["caption"], n_variants=n_variants)
             # A parità di immagini
            for v_i, variant_text in enumerate(variants):
                prompts.append({
                    # Aggiunge un nuovo record alla lista dei prompt sintetici.
                    **rec,
                    "variant_index": v_i, 
                    "variant_text": variant_text,
                    "prompt": f"a photo of a {rec['breed_name']}, {variant_text}",
                })
        return prompts

    def clear_gpu(self):
        """
        Liberazione di BLIP dalla memoria GPU
        """
        del self.model, self.tokenizer
        self.model, self.tokenizer = None, None
        clear_gpu_memory()

In [ ]:
class ImageGenerator:
    # Testo contenente caratteristiche visive da evitare nelle immagini generate.
    NEGATIVE_PROMPT = "blurry, distorted, deformed, extra limbs, low quality, watermark, text, cartoon"

    def __init__(self, model_name):
        self.model_name = model_name
        self.pipe = None

    def load_model(self):
        self.pipe = AutoPipelineForText2Image.from_pretrained(self.model_name, torch_dtype=dtype).to(device)
        self.pipe.set_progress_bar_config(disable=True)

    def generate(self, prompt, seed=None):
        """
        Genera una singola immagine sintetica da un prompt testuale.
        height/width=512: sdxl-turbo è stato ottimizzato per generare a 
        questa risoluzione; le immagini vengono comunque ridimensionate a 
        IMG_SIZE (224) subito dopo, la dimensione attesa dal classificatore.
        """
        generator = torch.Generator(device=device).manual_seed(seed) if seed is not None else None
        # synth_steps: numero di passi di ripulitura del rumore del modello di
        # diffusione -- piu' passi tendono a dare un'immagine piu' rifinita, a
        # costo di piu' tempo (ogni passo e' una chiamata completa al modello).
        # guidance_scale: quanto la generazione viene spinta a seguire fedelmente
        # il testo del prompt, invece di generare piu' liberamente.
        # generazione
        result = self.pipe(
            prompt=prompt,
            negative_prompt=self.NEGATIVE_PROMPT,
            num_inference_steps=cfg.SYNTH_STEPS,
            guidance_scale=0.0,
            height=512,
            width=512,
            generator=generator,
        )
        # prendo l'immagine e la ridimensiono
        return result.images[0].resize((cfg.IMG_SIZE, cfg.IMG_SIZE))

    def run_image_generation(self, prompts, synth_dir):
        """Genera e salva su disco un'immagine sintetica per ogni prompt."""
        os.makedirs(synth_dir, exist_ok=True)
        records = []
        for rec in tqdm(prompts, desc="Generazione immagini con il modello di diffusione"):
            class_dir = Path(synth_dir) / f"{rec['label']}_{rec['breed']}"
            class_dir.mkdir(parents=True, exist_ok=True)

            synth_image = self.generate(
                rec["prompt"],
                seed=SEED + rec["source_idx"] * 10 + rec["variant_index"]
            )
            save_path = class_dir / f"src{rec['source_idx']}_v{rec['variant_index']}.png"
            synth_image.save(save_path)
            records.append({
                "path": str(save_path),
                "label": rec["label"],
                "breed": rec["breed"],
                "source_idx": rec["source_idx"],
                "caption": rec["caption"],
                "variant_text": rec["variant_text"],
                "prompt": rec["prompt"],
            })
        return records

    def clear_gpu(self):
        del self.pipe
        self.pipe = None
        clear_gpu_memory()

In [ ]:
class ClipScorer:
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = None
        self.processor = None

    def load_model(self):
        self.model = CLIPModel.from_pretrained(self.model_name).to(device)
        # prepara in un colpo solo sia il testo (text=[text], tokenizzato) 
        # sia l'immagine (images=image, trasformata in tensore)
        self.processor = CLIPProcessor.from_pretrained(self.model_name)
        self.model.eval()

    @torch.no_grad()
    def score(self, image, text):
        """
        Calcola quanto un'immagine e' semanticamente coerente con un testo,
        tramite la similarita' coseno tra gli embedding di CLIP (image
        embedding e text embedding), moltiplicata per 100.
        """
        inputs = self.processor(text=[text], images=image,
                                 return_tensors="pt", padding=True).to(device)
        # passati gli input come argomenti e genera gli embedding per txt e img
        outputs = self.model(**inputs)
        # normalizzazione del vettore dell'img e txt a lunghezza 1
        # Serve perché la similarità coseno tra due vettori è, per definizione, 
        # il prodotto scalare tra le loro versioni normalizzate
        img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
        txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
        # prodotto scalare con estrazione del valore del tensore risultante
        # * 100 perchè la similarità coseno varia tipicamente in un range piccolo, tipo 0.25-0.40
        return (img_emb @ txt_emb.T).item() * 100 

    def score_dataframe(self, df):
        """
        Calcola il CLIP score per ogni riga di df (colonne "path" e "prompt"),
        restituendo lo stesso DataFrame con una colonna "clip_score" aggiunta.
        """
        scores = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Calcolo CLIP score"):
            img = Image.open(row["path"]).convert("RGB")
            scores.append(self.score(img, row["prompt"]))
        return df.assign(clip_score=scores)

    def clear_gpu(self):
        del self.model, self.processor
        self.model, self.processor = None, None
        clear_gpu_memory()

## 10. Pipeline di Data Augmentation generativa

Qui costruisco la parte centrale del progetto, nell'ordine richiesto dalla traccia:

1. **Image captioning**: per ogni immagine campionata per l'augmentation, genero una didascalia con BLIP.
2. **Generazione testuale**: genero `N_TEXT_VARIANTS` parafrasi della didascalia con un modello linguistico instruction-tuned, `Qwen2.5-1.5B-Instruct`.
3. **Generazione di immagini**: per ogni parafrasi genero una nuova immagine sintetica con un modello di diffusione, `sdxl-turbo`, condizionata dal testo.

Le tre fasi sono organizzate in **fasi separate** invece che in un unico ciclo che fa tutto insieme immagine per immagine: prima si generano tutte le caption con BLIP, poi si libera BLIP dalla memoria GPU e si generano tutte le varianti testuali con Qwen, poi si libera anche Qwen e si generano tutte le immagini sintetiche con il modello di diffusione. Il motivo non è solo di chiarezza espositiva: tenere contemporaneamente in memoria BLIP, un LLM da 1,5 miliardi di parametri e un modello di diffusione delle dimensioni di SDXL avrebbe un costo di VRAM inutilmente alto, dato che in realtà ciascun modello serve solo per la propria fase.

### Scelta di design

Non chiediamo a Qwen di modificare l'intero prompt contenente anche il nome della razza. Il modello riceve e rielabora solamente la parte descrittiva ottenuta da BLIP, relativa ad aspetti come posa, sfondo e contesto dell'immagine. Il nome della razza viene invece mantenuto separatamente, perché rappresenta la ground truth fornita direttamente dal dataset OxfordIIITPet.

Una volta ottenuta la variante testuale da Qwen, il prompt finale viene ricostruito tramite codice, combinando la razza originale, che non è stata modificata, con la nuova descrizione. In questo modo Qwen non può sostituire direttamente la classe con un'altra razza: la label viene mantenuta fuori dalla parte di testo affidata al modello linguistico e reinserita successivamente in modo deterministico.

Questa scelta riduce quindi alla fonte una possibile causa di label noise a livello testuale. 

### Campionamento del 30% del training da aumentare

Non applico la pipeline di captioning/generazione a tutto il training set: campiono, in modo stratificato con `train_test_split`, un sottoinsieme pari ad `AUGMENT_FRACTION` (30% di default) del training — la stessa proporzione richiesta dalla traccia per i dati sintetici. Con `N_TEXT_VARIANTS=1` variante generata per ogni immagine campionata, il numero di immagini sintetiche prodotte risulta quindi pari al 30% del training set (sopra la soglia minima del 20% richiesta dalla traccia).

In [ ]:
# ============================================================
# Campionamento del sottoinsieme da aumentare
# ============================================================
train_labels_array = trainval_labels[train_idx]

augment_idx, _ = train_test_split(
    train_idx,
    train_size=cfg.AUGMENT_FRACTION,
    # garantire che il AUGMENT_FRACTION% campionato per augment_idx 
    # mantenga le stesse proporzioni tra le razze che c'erano già in train_idx
    stratify=train_labels_array,
    random_state=SEED + 2,
)
augment_idx = sorted(augment_idx)

print(f"Training set: {len(train_idx)} immagini")
print(f"Sottoinsieme campionato per l'augmentation: {len(augment_idx)} immagini "
      f"({100 * len(augment_idx) / len(train_idx):.1f}% del training)")

### 10.1 Captioning con BLIP — sul sottoinsieme campionato per l'augmentation

In [ ]:
# ============================================================
# Caricamento del modello di captioning (BLIP)
# ============================================================
blip = Blip(cfg.CAPTION_MODEL)
blip.load_model()

In [ ]:
# ============================================================
# Captioning di tutte le immagini campionate per l'augmentation
# ============================================================
caption_records = blip.run_captioning(augment_idx)
print(f"Caption generate: {len(caption_records)}")

In [ ]:
caption_records[:10]

In [ ]:
# ============================================================
# Liberazione di BLIP dalla memoria GPU
# ============================================================
blip.clear_gpu()

### 10.2 Generazione testuale di varianti con Qwen2.5-1.5B-Instruct

`Qwen2.5-1.5B-Instruct` è un modello linguistico instruction-tuned: risponde a istruzioni in linguaggio naturale invece di limitarsi a completare del testo. Per generare le parafrasi uso il suo chat template (`apply_chat_template`), lo stesso meccanismo con cui il modello è stato addestrato a interagire, con un messaggio di sistema che gli chiede di restituire solo la frase riscritta, senza commenti aggiuntivi.

In [ ]:
# ============================================================
# Caricamento del modello di generazione testuale (Qwen2.5-1.5B-Instruct)
# ============================================================
qwen = Qwen(cfg.TEXT_GEN_MODEL)
qwen.load_model()

In [ ]:
# ============================================================
# Generazione delle varianti testuali per tutte le caption
# ============================================================
synth_prompts = qwen.run_variant_generation(caption_records, n_variants=cfg.N_TEXT_VARIANTS)
print(f"Prompt totali pronti per la generazione di immagini: {len(synth_prompts)}")

In [ ]:
synth_prompts[:10]

### Costruzione delle varianti testuali e dei prompt

Per ogni immagine selezionata viene generata inizialmente una **caption**, cioè una
descrizione testuale sintetica del contenuto visivo.

Successivamente, un modello generativo testuale produce una **variante della caption**
(`variant_text`) con formulazione diversa ma significato simile. Lo scopo è aumentare
la varietà linguistica dei prompt e ridurre la dipendenza da una singola descrizione.

La variante testuale viene poi combinata con il nome della classe reale (`breed_name`)
per costruire il **prompt finale** destinato al modello text-to-image.

Esempio:

Caption originale:
`a cat sitting on a blue bag`

Variante generata:
`A cat perches atop a blue tote bag.`

Prompt finale:
`a photo of a Abyssinian, A cat perches atop a blue tote bag.`

In questo modo il prompt conserva sia:
- l'informazione sulla **classe da generare**;
- il **contesto visivo** ricavato dall'immagine originale.

La struttura dei record mantiene inoltre il collegamento con l'immagine sorgente
tramite `source_idx` e con la classe tramite `label` e `breed`, così da poter
tracciare l'origine di ogni immagine sintetica generata.

In [ ]:
# ============================================================
# Liberazione di Qwen dalla memoria GPU
# ============================================================
qwen.clear_gpu()

### 10.3 Generazione di immagini sintetiche con sdxl-turbo

`stabilityai/sdxl-turbo` è la versione distillata di Stable Diffusion XL: eredita l'architettura e la qualità visiva di SDXL ma genera un'immagine in 1-4 step di diffusione invece dei 25-50 tipici di un modello di diffusione classico, il che lo rende compatibile con i tempi di una sessione Colab anche generando centinaia di immagini. 

In [ ]:
# ============================================================
# Carico il modello di generazione immagini
# ============================================================
image_gen = ImageGenerator(cfg.IMAGE_GEN_MODEL)
image_gen.load_model()

In [ ]:
# ============================================================
# Visualizzazione: reale vs una o piu sintetiche
# ============================================================
def show_real_vs_synthetic(rows):
    """
    Disegna una griglia con una riga per ogni elemento di rows.

    INPUT:
    - rows: lista di tuple (titolo, immagine_reale, lista_di_immagini_sintetiche).
    OUTPUT:
    - nessuno, mostra direttamente la figura con plt.show().
    """
    n_cols = max(1 + len(synth_imgs) for _, _, synth_imgs in rows)
    fig, axes = plt.subplots(len(rows), n_cols, figsize=(3.7 * n_cols, 3.3 * len(rows)), squeeze=False)

    for row_i, (title, real_img, synth_imgs) in enumerate(rows):
        axes[row_i, 0].imshow(real_img)
        axes[row_i, 0].set_title(f"{title} — reale", fontsize=9)
        axes[row_i, 0].axis("off")

        for col_i, synth_img in enumerate(synth_imgs, start=1):
            axes[row_i, col_i].imshow(synth_img)
            axes[row_i, col_i].set_title(f"{title} — sintetica {col_i}", fontsize=9)
            axes[row_i, col_i].axis("off")

        for col_i in range(1 + len(synth_imgs), n_cols):
            axes[row_i, col_i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# TEST — Generazione singola immagine
# ============================================================
_demo_record = synth_prompts[0]
_demo_real_img, _ = trainval_raw[_demo_record["source_idx"]]
_demo_synth = image_gen.generate(_demo_record["prompt"], seed=SEED)

show_real_vs_synthetic([(_demo_record["breed_name"], _demo_real_img, [_demo_synth])])
print(f"Prompt usato: \"{_demo_record['prompt']}\"")

### Generazione dell'intero dataset sintetico

Ora ripeto il procedimento appena testato su un solo esempio per **tutti** i prompt preparati al passo precedente, salvando ogni immagine sintetica su disco insieme al prompt usato e alla classe di appartenenza (ereditata dalla classe dell'immagine reale di partenza — è l'unica etichetta che ha senso usare, dato che il prompt include esplicitamente quella razza).

In [ ]:
# ============================================================
# Generazione di tutte le immagini sintetiche
# ============================================================
print(f"Immagini sintetiche da generare: {len(synth_prompts)}")
synth_records = image_gen.run_image_generation(synth_prompts, cfg.SYNTH_DIR)

In [ ]:
# Visualizzaziamo le info di salvataggio delle immagini generate
synth_df = pd.DataFrame(synth_records)
synth_df.to_csv(Path(cfg.SYNTH_DIR) / "synthetic_metadata.csv", index=False)
print(f"Generate {len(synth_df)} immagini sintetiche su {len(synth_prompts)} attese.")
synth_df.head()

In [ ]:
# ============================================================
# Liberazione del modello di generazione immagini dalla memoria GPU
# ============================================================
image_gen.clear_gpu()

## 11. Valutazione della qualità dei dati sintetici

Prima di usare le immagini sintetiche per il training, ha senso chiedersi: sono davvero coerenti con il testo che le ha generate? Per rispondere in modo più oggettivo di un controllo visivo a campione, uso il **CLIP score**: la similarità coseno tra l'embedding dell'immagine e l'embedding del testo calcolati da un modello CLIP pre-addestrato, moltiplicata per 100 (è una versione semplificata della metrica proposta da Hessel et al., 2021, "CLIPScore: A Reference-free Evaluation Metric for Image Captioning"). Valori più alti indicano maggiore coerenza semantica tra immagine e testo.

In [ ]:
# ============================================================
# Caricamento CLIP e calcolo del CLIP score sui sintetici
# ============================================================
clip_scorer = ClipScorer(cfg.QUALITY_CHECK_MODEL)
clip_scorer.load_model()

sample_size = len(synth_df)
# Estrae casualmente sample_size righe dal DataFrame delle immagini sintetiche.
clip_sample = synth_df.sample(n=sample_size, random_state=SEED)
clip_sample = clip_scorer.score_dataframe(clip_sample)
scores = clip_sample["clip_score"].tolist()

print(f"CLIP score medio: {np.mean(scores):.2f}")
print(f"CLIP score mediano: {np.median(scores):.2f}")
print(f"CLIP score minimo/massimo: {np.min(scores):.2f} / {np.max(scores):.2f}")

clip_scorer.clear_gpu()

In [ ]:
# ============================================================
# Distribuzione del CLIP score
# ============================================================
plt.figure(figsize=(9, 4.5))
plt.hist(scores, bins=25, color="#55A868", edgecolor="white")
plt.axvline(np.mean(scores), color="black", linestyle="--", label=f"media = {np.mean(scores):.1f}")
plt.xlabel("CLIP score (immagine sintetica vs. prompt usato per generarla)")
plt.ylabel("Numero di immagini")
plt.title("Distribuzione del CLIP score sulle immagini sintetiche")
plt.legend()
plt.tight_layout()
plt.show()

## Valutazione delle immagini sintetiche tramite CLIP score

### Valori del CLIP score

- **CLIP score medio:** 35.16
- **CLIP score mediano:** 35.17
- **Range:** 28.37 – 43.87

Media e mediana sono quasi identiche, quindi i punteggi risultano abbastanza equilibrati e il valore medio non sembra essere influenzato in modo importante da pochi valori estremi.

Il CLIP score viene utilizzato per misurare la **similarità semantica tra ogni immagine sintetica e il prompt utilizzato per generarla**: a valori più alti corrisponde una maggiore somiglianza tra il contenuto dell'immagine e quello descritto dal prompt.

> **Nota:** il CLIP score non è una percentuale e non verifica direttamente se la razza rappresentata nell'immagine è quella corretta. In questo caso viene calcolato a partire dalla similarità coseno tra gli embedding dell'immagine e del testo, moltiplicata per 100.

### Distribuzione dei CLIP score

Dal grafico si vede che:

- la maggior parte dei punteggi è concentrata circa tra **32 e 39**;
- il centro della distribuzione si trova intorno a **35**, in accordo con la media e la mediana calcolate;
- sono presenti pochi valori molto lontani dalla zona centrale;
- alcune immagini hanno score più bassi, fino ad un minimo di **28.37**, mentre il valore massimo raggiunge **43.87**.

Nel complesso, la distribuzione appare abbastanza concentrata intorno al valore medio e non si osserva un gruppo numeroso di immagini con score molto più basso rispetto alle altre.

→ Il CLIP score fornisce quindi una prima valutazione automatica della **coerenza tra immagini sintetiche e relativi prompt**. Le immagini con score più basso possono essere controllate visivamente per capire se presentano effettivamente una minore corrispondenza con il prompt o eventuali problemi nella generazione.

In [ ]:
clip_sample

In [ ]:
# ============================================================
# Ispezione visiva delle sintetiche con CLIP score piu basso
# ============================================================
n_lowest = 6
lowest_scores = clip_sample.sort_values("clip_score").head(n_lowest)

fig, axes = plt.subplots(1, n_lowest, figsize=(3 * n_lowest, 3.8))
for ax, (_, row) in zip(axes, lowest_scores.iterrows()):
    img = Image.open(row["path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{row['breed'].replace('_', ' ')}\nCLIP score: {row['clip_score']:.1f}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Le {n_lowest} immagini sintetiche con CLIP score piu basso")
plt.tight_layout()
plt.show()

for _, row in lowest_scores.iterrows():
    print(f"CLIP score {row['clip_score']:.1f} — {row['breed'].replace('_', ' ')} — prompt: \"{row['prompt']}\"")

### Le sintetiche con CLIP score più basso

Queste sono le sintetiche col punteggio di coerenza più basso in assoluto — il caso peggiore che la pipeline ha prodotto.

- Guardandole, le immagini restano visivamente realistiche e mostrano comunque la razza corretta (il Pug è chiaramente un pug, il Siamese ha gli occhi blu tipici della razza). Il punteggio basso non segnala quindi un problema di identità della razza.
- Il pattern comune è piuttosto una **corrispondenza solo parziale col prompt**: alcuni dettagli descritti nel testo non compaiono nell'immagine generata (es. "due gatti accanto a un piccolo roditore" mostra solo i due gatti, senza roditore; "un gatto appollaiato sulla spalla di una donna" mostra solo il gatto, senza la donna). Il modello di diffusione sembra aver ignorato la parte più insolita del prompt, mantenendo comunque il soggetto principale.
- Anche il punteggio più basso (28,4) resta comunque nell'ordine di grandezza tipico di coppie immagine-testo coerenti, lontano dai valori tipici di coppie scollegate.

→ Qui il caso limite non è un problema di razza sbagliata o immagine scadente, ma di corrispondenza incompleta con un prompt più articolato — un limite del modello di diffusione con prompt insoliti, non della pipeline di etichettatura.

In [ ]:
# ============================================================
# Ispezione visiva: reale vs sintetiche, per alcune classi
# ============================================================
sample_classes = sorted(synth_df["label"].unique())[:4]

rows = []
for label in sample_classes:
    breed = CLASSES[label].replace("_", " ")
    real_idx = [i for i in augment_idx if trainval_raw[i][1] == label][0]
    real_img, _ = trainval_raw[real_idx]
    synth_rows = synth_df[synth_df["label"] == label].head(2)
    synth_imgs = [Image.open(p).convert("RGB") for p in synth_rows["path"]]
    rows.append((breed, real_img, synth_imgs))

show_real_vs_synthetic(rows)

### Osservazioni

- Le immagini sintetiche mostrate risultano generalmente **realistiche e coerenti con le rispettive classi**.
- Sono presenti variazioni di **posa, sfondo, illuminazione e inquadratura**, utili per aumentare la varietà del training set.
- Abyssinian e American Bulldog risultano particolarmente riconoscibili negli esempi osservati.
- Per American Pit Bull Terrier emerge maggiormente la possibile somiglianza con razze morfologicamente vicine.

→ Il controllo visivo completa il CLIP score: un'immagine può essere coerente con il prompt ma non rappresentare perfettamente la classe assegnata.

## 12. Esperimento B — Augmented (dati reali + sintetici)

### Costruzione del dataset aumentato

Il dataset aumentato è l'unione di `train_dataset` (le stesse immagini reali del baseline, invariate) e di un dataset che carica le immagini sintetiche appena generate, con la stessa trasformazione di training. Uso `ConcatDataset`, che unisce due dataset PyTorch senza copiare i dati in memoria.

In [ ]:
# ============================================================
# Dataset delle immagini sintetiche
# ============================================================
class SyntheticImageDataset(Dataset):
    def __init__(self, metadata_df, transform):
        self.df = metadata_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        return self.transform(image), int(row["label"])


synthetic_dataset = SyntheticImageDataset(synth_df, train_transform)
augmented_train_dataset = ConcatDataset([train_dataset, synthetic_dataset])

print(f"Training baseline:  {len(train_dataset)} immagini (solo reali)")
print(f"Training augmented: {len(augmented_train_dataset)} immagini "
      f"({len(train_dataset)} reali + {len(synthetic_dataset)} sintetiche)")

augmented_train_loader = DataLoader(augmented_train_dataset, 
                                    batch_size=cfg.BATCH_SIZE,
                                    shuffle=True, 
                                    num_workers=2,
                                    persistent_workers=True)

In [ ]:
# ============================================================
# Distribuzione delle classi nel dataset aumentato
# ============================================================
augmented_labels = [train_dataset[i][1] for i in range(len(train_dataset))] + \
                    synth_df["label"].tolist()
augmented_counts = pd.Series(augmented_labels).value_counts().sort_index()

class_distribution(
    NUM_CLASSES, 
    class_counts, 
    "trainval_augmented",
    class_counts_2=augmented_counts,
    label_1="Trainval completo (riferimento)",
    label_2="Training aumentato (reale + sintetico)",
    title="Dimensione del training set per classe: aumentato vs. trainval completo",
)

### Distribuzione delle classi: training aumentato vs. trainval completo

Il grafico confronta, per ciascuna razza, quante immagini ha ora il training aumentato (reali + sintetiche) rispetto all'intero trainval originale.

- La barra più chiara (trainval completo) è presa solo come **riferimento visivo**, non come obiettivo da raggiungere.
- Il training aumentato è composto dall'80% del trainval usato per il training, più un ulteriore 30% di immagini sintetiche: è normale che resti sotto il trainval completo per la maggior parte delle classi.
- Ogni classe cresce della **stessa proporzione**, perché sia lo split train/validation sia il campionamento per l'augmentation erano stratificati.

→ Qualunque differenza di prestazioni tra classi nella valutazione finale non è quindi dovuta a uno sbilanciamento introdotto da questo passaggio, ma va cercata altrove.

In [ ]:
# ============================================================
# Training del modello Augmented
# ============================================================
# Stessa architettura (build_model), stessi learning rate differenziati,
# stesso numero massimo di epoche, stessa policy di early stopping e stesso
# validation set reale del baseline. L'unica differenza rispetto
# all'esperimento precedente è il train_loader: augmented_train_loader al
# posto di baseline_train_loader.
augmented_model = build_model()
augmented_trainer = Trainer(
    augmented_model, 
    augmented_train_loader,
    val_loader,
    epochs=cfg.EPOCHS, 
    lr_backbone=cfg.LR_BACKBONE, 
    lr_head=cfg.LR_HEAD,
    patience=cfg.PATIENCE, 
    min_delta=cfg.MIN_DELTA,
    label="augmented",
)
augmented_model, augmented_history = augmented_trainer.fit()

In [ ]:
# ============================================================
# Curve di training del modello Augmented
# ============================================================
augmented_evaluator = Evaluator(augmented_model, CLASSES)
augmented_evaluator.plot_history(augmented_history, "Esperimento Augmented")

### Analisi delle curve di training — Augmented

In questo esperimento il modello viene addestrato sul **training set aumentato con le immagini sintetiche generate**.

Dalle curve si vede che il modello **impara velocemente già nelle prime epoche**. La training loss scende da circa **2.90 a 1.15 nelle prime 6 epoche**, mentre la training accuracy passa dal **30,8% a circa l'87,6%**.

Continuando il training, il modello migliora progressivamente sui dati di training: nelle ultime epoche la **training accuracy arriva a circa il 99,6%**, mentre la training loss continua a diminuire fino a valori intorno a **0,76-0,79**.

Anche sul validation set si osserva un miglioramento progressivo. La **validation accuracy parte dal 67,1%** e supera il **90% già intorno all'epoca 13-14**, continuando poi a migliorare più lentamente, con qualche oscillazione. Il valore più alto osservato è circa **93,3%, intorno all'epoca 51**.

La validation loss segue un andamento simile: diminuisce molto nelle prime epoche e successivamente più lentamente, scendendo sotto **0,94** nelle epoche finali, fino a un valore migliore di circa **0,937 intorno all'epoca 51**. Dopo questo punto oscilla leggermente senza ottenere un nuovo miglioramento sufficiente.

Anche in questo caso si crea progressivamente una distanza tra training e validation: il modello arriva quasi al **100% di accuracy sul training**, mentre la validation si mantiene intorno al **92-93,3%**. È quindi presente un certo **overfitting**, ma le prestazioni sul validation set rimangono comunque elevate.

L'**early stopping** interrompe il training all'epoca **55**, dopo 4 epoche senza un miglioramento sufficiente della validation loss. In questo modo il training si ferma prima di raggiungere le 80 epoche massime previste.

> **Nota:** anche in questo esperimento viene utilizzato il **label smoothing**, quindi la loss non deve necessariamente avvicinarsi a zero quando l'accuracy diventa molto alta. Per questo una training loss intorno a **0,76-0,79** è compatibile con una training accuracy vicina al **100%**.

→ Nel complesso, il modello Augmented apprende bene anche utilizzando il dataset arricchito con immagini sintetiche. La validation continua a migliorare fino a circa metà dell'ultimo terzo del training e raggiunge valori superiori al 93%. Il passo successivo sarà confrontare questi risultati con il Baseline sullo **stesso test set**, per verificare se l'aggiunta delle immagini sintetiche porta un miglioramento anche su dati mai utilizzati durante il training.

### Primo confronto con il Baseline

Confrontando le curve con quelle ottenute dal Baseline si osservano alcune differenze:

- il **Baseline** raggiunge una migliore validation loss di circa **0,989**, mentre l'**Augmented scende fino a circa 0,937**;
- la migliore validation accuracy passa da circa **90,8% nel Baseline a circa 93,3% nell'Augmented**;
- il Baseline viene fermato dall'early stopping all'**epoca 47**, mentre l'Augmented continua fino all'**epoca 55**, mostrando miglioramenti della validation loss per un tratto più lungo.

Quindi, già durante la fase di validation, il modello addestrato con le immagini sintetiche mostra **risultati migliori rispetto al Baseline**.

→ Questo primo confronto suggerisce che l'aggiunta delle immagini sintetiche possa aver aiutato la capacità di generalizzazione del modello. La conferma più importante deve però arrivare dal confronto dei due modelli sullo **stesso test set reale**, che permette di verificare se il vantaggio osservato in validation si mantiene anche su dati mai utilizzati durante il training.

## 13. Valutazione dell'esperimento Augmented sul test set

Stesso test set, stesse metriche, stesso codice di valutazione del baseline: cambia solo il modello.

In [ ]:
# ============================================================
# Metriche Augmented sul test set
# ============================================================
augmented_results = augmented_evaluator.evaluate(test_loader)

print(f"Accuracy:            {augmented_results['accuracy']:.4f}")
print(f"Top-3 accuracy:       {augmented_results['top3_accuracy']:.4f}")
print(f"Precision (macro):   {augmented_results['precision_macro']:.4f}")
print(f"Recall (macro):      {augmented_results['recall_macro']:.4f}")
print(f"F1 (macro):          {augmented_results['f1_macro']:.4f}")
print(f"F1 (weighted):       {augmented_results['f1_weighted']:.4f}")
print(augmented_results['report'])

In [ ]:
# ============================================================
# Matrice di confusione Augmented
# ============================================================
augmented_evaluator.plot_confusion_matrix(
    augmented_results["confusion_matrix"], "Matrice di confusione — Augmented (test set)"
)

In [ ]:
# ============================================================
# Classi più difficili per il modello Augmented
# ============================================================

hardest_baseline = evaluator.hardest_classes(
    augmented_results,
    top_n=10
)

hardest_baseline

In [ ]:
# ============================================================
# Confusioni più frequenti del modello Augmented
# ============================================================

confusions_baseline = evaluator.most_confused_classes(
    augmented_results,
    top_n=15
)

confusions_baseline

### Valutazione del modello Augmented sul test set

Il modello Augmented, addestrato sul training set arricchito con le **immagini sintetiche generate**, raggiunge sul test set una **Accuracy del 90,54%**. Questo significa che circa 91 immagini su 100 vengono classificate correttamente come prima scelta.

La **Top-3 Accuracy raggiunge il 97,19%**: nella maggior parte dei casi la classe corretta si trova quindi almeno tra le tre classi considerate più probabili dal modello.

Le altre metriche ottenute sono:

- **Precision Macro: 90,82%**
- **Recall Macro: 90,50%**
- **F1 Macro: 90,41%**
- **F1 Weighted: 90,49%**

Precision, Recall e F1 hanno valori molto vicini tra loro, quindi il comportamento complessivo del modello risulta abbastanza equilibrato.

Anche **F1 Macro e F1 Weighted sono quasi uguali**, quindi il risultato complessivo cambia molto poco considerando tutte le classi allo stesso modo oppure pesandole in base al loro numero di esempi.

### Matrice di confusione e classi più difficili

Osservando la matrice di confusione si nota una **diagonale molto evidente**, quindi per la maggior parte delle 37 razze molte immagini vengono classificate correttamente.

Analizzando però le singole classi, alcune risultano più difficili da riconoscere.

La classe più problematica è **American Pit Bull Terrier**, con:

- **Precision: 86,21%**
- **Recall: 50,00%**
- **F1-score: 63,29%**

Il recall del 50% significa che, tra le immagini realmente appartenenti a questa razza, il modello ne riconosce correttamente **1 su 2**.

Tra le altre classi più difficili troviamo:

- **Staffordshire Bull Terrier:** precision 57,41%, recall ≈ **69,66%**, F1 ≈ **62,94%**
- **Ragdoll:** recall **74,00%**, F1 ≈ **76,29%**
- **British Shorthair:** recall **79,00%**, F1 ≈ **85,87%**
- **Egyptian Mau:** recall ≈ **83,51%**, F1 ≈ **87,10%**
- **Persian:** recall **85,00%**, F1 ≈ **86,73%**
- **Birman:** recall **86,00%**, F1 ≈ **82,69%**
- **Maine Coon:** recall **86,00%**, F1 ≈ **86,87%**

Quindi, anche se le prestazioni complessive sono alte, il modello continua ad avere maggiori difficoltà nel riconoscere alcune razze specifiche.

### Confusioni più frequenti

La confusione più frequente è:

- **American Pit Bull Terrier → Staffordshire Bull Terrier: 30 errori**

Seguono:

- **Egyptian Mau → Bengal: 15 errori**
- **Ragdoll → Birman: 15 errori**
- **American Pit Bull Terrier → American Bulldog: 11 errori**
- **Staffordshire Bull Terrier → American Bulldog: 11 errori**
- **Birman → Ragdoll: 10 errori**
- **British Shorthair → Russian Blue: 9 errori**
- **Basset Hound → Beagle: 7 errori**
- **Bengal → Egyptian Mau: 7 errori**
- **Siamese → Birman: 7 errori**

Anche dopo l'aggiunta delle immagini sintetiche rimangono quindi alcune confusioni tra specifiche coppie di razze. Ad esempio, `American Pit Bull Terrier` viene confuso soprattutto con `Staffordshire Bull Terrier` e `American Bulldog`.

Si osserva inoltre una confusione in entrambe le direzioni tra `Ragdoll` e `Birman`: **15 Ragdoll vengono classificati come Birman e 10 Birman vengono classificati come Ragdoll**.

### Considerazioni sul modello Augmented

Nel complesso, il modello Augmented raggiunge **buone prestazioni sul test set reale**, con un'Accuracy del 90,54% e una Top-3 Accuracy del 97,19%.

L'analisi per classe mostra però che l'aggiunta delle immagini sintetiche **non elimina completamente le difficoltà sulle classi più complesse**. Alcune razze continuano infatti ad avere un recall più basso e alcune coppie vengono confuse frequentemente.

Per capire quale sia stato l'effetto delle immagini sintetiche non è però sufficiente osservare solamente i risultati del modello Augmented: è necessario confrontarli direttamente con quelli ottenuti dal **Baseline**, sia nelle metriche globali sia nelle prestazioni delle singole classi e nelle principali confusioni.

## 14. Confronto finale tra Baseline e Augmented

Metto ora insieme tutti i risultati calcolati finora. Oltre alle metriche già viste, aggiungo due analisi pensate apposta per il confronto tra i due esperimenti:

- il **delta di recall per classe**, per capire se il beneficio (o il danno) dell'augmentation è distribuito uniformemente tra le razze o concentrato su alcune;
- un **intervallo di confidenza bootstrap al 95%** sulla differenza di accuracy tra i due modelli sullo stesso test set, per avere un'idea di quanto la differenza osservata sia solida e non semplicemente rumore di campionamento — un solo numero di differenza, senza intervallo, può essere fuorviante quando il test set è di dimensione finita.

In [ ]:
# ============================================================
# Funzione: tabella comparativa delle metriche principali
# ============================================================
METRIC_KEYS = {
    "Accuracy": "accuracy",
    "Top-3 accuracy": "top3_accuracy",
    "Precision (macro)": "precision_macro",
    "Recall (macro)": "recall_macro",
    "F1 (macro)": "f1_macro",
    "F1 (weighted)": "f1_weighted",
}


def build_comparison_table(results_by_name, metrics=None):
    """
    Costruisce una tabella comparativa a partire da un dizionario
    {nome_modello: dizionario_risultati}, dove ogni dizionario_risultati
    e' il ritorno di Evaluator.evaluate().
    """
    if metrics is None:
        metrics = list(METRIC_KEYS.keys())
    return pd.DataFrame({
        name: {label: results[METRIC_KEYS[label]] for label in metrics}
        for name, results in results_by_name.items()
    }).round(4)

In [ ]:
# ============================================================
# Tabella comparativa delle metriche principali
# ============================================================
comparison = build_comparison_table({
    "Baseline (solo reali)": baseline_results,
    "Augmented (reali + sintetiche)": augmented_results,
})
comparison["Delta"] = (comparison["Augmented (reali + sintetiche)"]
                        - comparison["Baseline (solo reali)"]).round(4)
comparison

In [ ]:
# ============================================================
# Grafico comparativo delle metriche principali
# ============================================================
metrics_to_plot = ["Accuracy", "Top-3 accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width / 2, comparison.loc[metrics_to_plot, "Baseline (solo reali)"], width, label="Baseline")
ax.bar(x + width / 2, comparison.loc[metrics_to_plot, "Augmented (reali + sintetiche)"], width, label="Augmented")
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.set_ylabel("Valore della metrica")
ax.set_title("Confronto Baseline vs. Augmented sul test set")
ax.legend()
plt.tight_layout()
plt.show()

### Confronto tra Baseline e Augmented

In questa fase vengono confrontati i due modelli sullo **stesso test set reale**:

- **Baseline:** addestrato solamente con immagini reali;
- **Augmented:** addestrato con immagini reali + immagini sintetiche generate.

Dalla tabella e dal grafico si vede che il modello **Augmented ottiene risultati migliori del Baseline in tutte le metriche considerate**:

| Metrica | Baseline | Augmented | Miglioramento |
|---|---:|---:|---:|
| Accuracy | 88,66% | 90,54% | **+1,88 punti** |
| Top-3 Accuracy | 97,11% | 97,19% | **+0,08 punti** |
| Precision Macro | 88,96% | 90,82% | **+1,86 punti** |
| Recall Macro | 88,61% | 90,50% | **+1,89 punti** |
| F1 Macro | 88,45% | 90,41% | **+1,96 punti** |
| F1 Weighted | 88,52% | 90,49% | **+1,97 punti** |

Il miglioramento non riguarda quindi solamente l'Accuracy, ma anche **Precision, Recall e F1**. In particolare, il miglioramento del **Macro F1 di circa 1,96 punti** è interessante perché questa metrica considera allo stesso modo tutte le classi, senza dare più importanza a quelle con più esempi.

Anche la **Top-3 Accuracy** aumenta, ma di molto meno (dal 97,11% al 97,19%, solo +0,08 punti): il valore del Baseline era già molto alto, quindi il margine disponibile per migliorare era minimo.

Guardando anche l'analisi precedente delle singole classi, il miglioramento non è comunque identico per tutte le razze. Ad esempio, `American Pit Bull Terrier` rimane una delle classi più difficili, ma il suo **recall passa dal 42% al 50%** e il suo **F1-score dal 55,26% al 63,29%**. Allo stesso tempo, alcune confusioni specifiche rimangono presenti, come quella con `Staffordshire Bull Terrier`.

→ Nel complesso, questi risultati indicano che l'aggiunta delle **immagini sintetiche al training set è associata a un miglioramento delle prestazioni sul test set reale**. Il modello Augmented supera infatti il Baseline in tutte le metriche considerate, suggerendo che i dati sintetici hanno fornito informazioni utili durante l'addestramento e hanno migliorato la capacità di generalizzazione del modello.

In [ ]:
# ============================================================
# Delta di recall per classe: dove l'augmentation aiuta di più
# ============================================================
recall_baseline_per_class = np.array([
    baseline_results["report"][c]["recall"] for c in CLASSES
])
recall_augmented_per_class = np.array([
    augmented_results["report"][c]["recall"] for c in CLASSES
])
recall_delta = recall_augmented_per_class - recall_baseline_per_class

order = np.argsort(recall_delta)
sorted_classes = np.array(CLASSES)[order]
sorted_delta = recall_delta[order]

plt.figure(figsize=(10, 14))
colors = ["#C44E52" if d < 0 else "#55A868" for d in sorted_delta]
plt.barh([c.replace("_", " ") for c in sorted_classes], sorted_delta, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Delta recall (Augmented - Baseline)")
plt.title("Variazione del recall per razza, ordinata dal peggioramento al miglioramento maggiore")
plt.tight_layout()
plt.show()

print(f"Classi migliorate: {(recall_delta > 0).sum()} / {NUM_CLASSES}")
print(f"Classi peggiorate: {(recall_delta < 0).sum()} / {NUM_CLASSES}")
print(f"Classi invariate: {(recall_delta == 0).sum()} / {NUM_CLASSES}")

### Variazione del recall per classe — Baseline vs Augmented

Questo grafico mostra come cambia il **recall di ogni razza** passando dal modello Baseline al modello Augmented.

- Le barre **verdi** indicano le classi in cui il recall è migliorato con l'aggiunta delle immagini sintetiche.
- Le barre **rosse** indicano invece le classi in cui il recall è diminuito.
- Le classi senza una variazione visibile hanno ottenuto praticamente lo stesso recall nei due modelli.

Nel complesso:

- **24 classi su 37 migliorano**
- **5 classi su 37 peggiorano**
- **8 classi su 37 rimangono invariate**

Quindi l'effetto delle immagini sintetiche non è uguale per tutte le razze, ma il numero di classi che migliorano è nettamente maggiore rispetto a quelle che peggiorano.

Tra i miglioramenti più evidenti troviamo:

- **Maine Coon:** il miglioramento più grande in assoluto, recall dal 77% all'86% (**+9 punti**)
- **American Pit Bull Terrier:** dal 42% al 50% (**+8 punti**) — la classe più difficile del Baseline
- **Miniature Pinscher** e **Abyssinian:** tra i miglioramenti più marcati, subito dopo i due precedenti
- **Ragdoll:** dal 69% al 74% (**+5 punti**)
- **Persian:** dall'81% all'85% (**+4 punti**)
- **Egyptian Mau:** dall'80% all'84% circa (**+3 punti**)
- **Staffordshire Bull Terrier:** dal 67% al 70% circa (**+2 punti**) — un miglioramento più contenuto di quanto ci si aspetterebbe data la sua difficoltà

È interessante soprattutto il miglioramento di `American Pit Bull Terrier`, perché nel Baseline era risultata la classe con maggiori difficoltà. Anche `Maine Coon`, che pure non era tra le classi più difficili in assoluto, mostra il miglioramento percentualmente più grande.

Non tutte le classi però beneficiano dell'augmentation. Le uniche 5 classi che peggiorano sono, dalla più alla meno colpita:

- **Boxer** (il peggioramento maggiore, comunque contenuto: circa **-2 punti**)
- **Bengal**
- **Great Pyrenees**
- **Pomeranian**
- **Yorkshire Terrier** (il peggioramento più lieve tra i cinque)

Anche il peggioramento più marcato (Boxer) resta di entità modesta rispetto ai miglioramenti più grandi.

→ Nel complesso, l'aggiunta delle immagini sintetiche ha un **effetto chiaramente positivo sul recall della maggior parte delle classi**: 24 su 37 migliorano, contro appena 5 che peggiorano (in modo comunque contenuto). Il risultato più interessante è che le classi che nel Baseline erano più difficili, come `American Pit Bull Terrier`, migliorano in modo evidente — anche se non sono le uniche né le più grandi tra le classi che beneficiano dell'augmentation.

In [ ]:
# ============================================================
# Intervallo di confidenza bootstrap sulla differenza di accuracy
# ============================================================
# La differenza di accuracy osservata sul test set è un singolo valore
# e potrebbe dipendere, almeno in parte, dalle specifiche immagini
# presenti nel test set.
#
# Il bootstrap permette di stimare quanto questa differenza sia stabile.
# ============================================================
def bootstrap_accuracy_diff(labels, preds_a, preds_b, n_bootstrap=2000, seed=SEED):
    """
    Confronta l'accuracy di due modelli tramite il metodo bootstrap.

    A partire dallo stesso test set vengono creati molti campioni casuali
    con reinserimento. Per ogni campione viene calcolata l'accuracy dei
    due modelli e salvata la loro differenza:
        accuracy modello B - accuracy modello A
    Ripetendo questa operazione molte volte si ottiene una distribuzione
    delle differenze di accuracy, che può essere usata successivamente
    per calcolare un intervallo di confidenza.

    INPUT:
    - labels: array contenente le etichette reali del test set.
    - preds_a: predizioni del primo modello, ad esempio la Baseline.
    - preds_b: predizioni del secondo modello, ad esempio l'Augmented.
    - n_bootstrap: numero di ricampionamenti bootstrap da eseguire.
    - seed: seme casuale per rendere il risultato riproducibile.
    OUTPUT:
    - diffs: array contenente, per ogni ricampionamento bootstrap,
      la differenza di accuracy tra modello B e modello A.
    """
    # Creo un generatore di numeri casuali inizializzato con il seed.
    rng = np.random.default_rng(seed)
    n = len(labels)
    # Creo un array vuoto che conterrà una differenza di accuracy
    # per ciascuno dei n_bootstrap ricampionamenti.
    diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        # Estraggo n indici casuali compresi tra 0 e n-1.
        # L'estrazione avviene con reinserimento:
        # uno stesso indice può quindi comparire più volte.
        idx = rng.integers(0, n, n)
        acc_a = (preds_a[idx] == labels[idx]).mean()
        acc_b = (preds_b[idx] == labels[idx]).mean()
        diffs[i] = acc_b - acc_a
    return diffs

In [ ]:
assert np.array_equal(baseline_results["labels"], augmented_results["labels"]), \
    "I due modelli devono essere valutati sullo stesso test set nello stesso ordine."

diffs = bootstrap_accuracy_diff(
    baseline_results["labels"], 
    baseline_results["preds"], 
    augmented_results["preds"]
)

# Per riassumere la variabilità dei risultati utilizziamo un
# intervallo di confidenza bootstrap al 95%.
# Viene quindi considerato il 95% centrale dei valori presenti in "diffs":
#     - il 2,5° percentile è il valore sotto cui si trova
#       il 2,5% delle differenze bootstrap;
#     - il 97,5° percentile è il valore sotto cui si trova
#       il 97,5% delle differenze bootstrap.
#
# In questo modo viene escluso il 2,5% dei valori più bassi e
# il 2,5% dei valori più alti, mantenendo il 95% centrale:
#        2,5%        |--------- 95% ---------|        2,5%
#                    ↑                       ↑
#                  ci_low                  ci_high
# Questo intervallo permette quindi di valutare l'incertezza
# associata alla differenza di accuracy osservata.
#
# In particolare, lo zero rappresenta "nessuna differenza"
# tra Augmented e Baseline:
# - se l'intervallo è > 0 → Augmented ottiene un'accuracy maggiore;
# - se l'intervallo è = 0 → i due modelli ottengono la stessa accuracy;
# - differenza < 0  → Baseline ottiene un'accuracy maggiore.
# -------
# L'intervallo di confidenza permette di considerare anche l'incertezza
# della differenza osservata:
# - se tutto l'intervallo è maggiore di 0, anche considerando la
#   variabilità stimata con il bootstrap, la differenza rimane positiva
#   e i risultati supportano un vantaggio dell'Augmented;
# - se l'intervallo comprende 0, tra i valori compatibili con la
#   variabilità stimata c'è anche "nessuna differenza" tra i modelli.
#   In questo caso il miglioramento osservato sul test set è meno
#   convincente, perché potrebbe non essere sufficientemente stabile.

ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
observed_diff = augmented_results["accuracy"] - baseline_results["accuracy"]

print(f"Differenza di accuracy osservata (Augmented - Baseline): {observed_diff:.4f}")
print(f"Intervallo di confidenza bootstrap al 95%: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"L'intervallo esclude lo zero: {'sì' if ci_low > 0 or ci_high < 0 else 'no'}")

In [ ]:
# ============================================================
# Distribuzione bootstrap della differenza di accuracy
# ============================================================
plt.figure(figsize=(9, 4.5))
plt.hist(diffs, bins=40, color="#4C72B0", edgecolor="white")
plt.axvline(0, color="black", linestyle="-", linewidth=1, label="nessuna differenza")
plt.axvline(observed_diff, color="#C44E52", linestyle="--", label=f"differenza osservata = {observed_diff:.4f}")
plt.axvspan(ci_low, ci_high, color="orange", alpha=0.15, label="IC 95%")
plt.xlabel("Differenza di accuracy (Augmented - Baseline) sui campioni bootstrap")
plt.ylabel("Frequenza")
plt.title("Distribuzione bootstrap della differenza di accuracy")
plt.legend()
plt.tight_layout()
plt.show()

### Analisi bootstrap della differenza di Accuracy

Dal confronto precedente abbiamo osservato che il modello Augmented raggiunge un'Accuracy maggiore rispetto al Baseline. Per capire se questo miglioramento è abbastanza stabile, viene utilizzato il **bootstrap**.

La differenza osservata sul test set originale è:

- **Differenza di Accuracy: +0,0188**, cioè circa **+1,88 punti percentuali**
- **Intervallo di confidenza bootstrap al 95%: [0,0112, 0,0264]**

Questo significa che, nei campioni bootstrap, il miglioramento dell'Augmented è compatibile con un valore compreso indicativamente tra **+1,12 e +2,64 punti percentuali**.

Nel grafico:

- la linea nera a **0** rappresenta il caso in cui Baseline e Augmented hanno la stessa Accuracy;
- la linea rossa tratteggiata rappresenta la differenza realmente osservata, pari a **+0,0188**;
- l'area evidenziata rappresenta l'**intervallo di confidenza al 95%**;
- l'istogramma mostra come si distribuiscono le differenze di Accuracy ottenute nei diversi campioni bootstrap.

La cosa più importante da osservare è che **l'intervallo di confidenza non contiene lo zero**: anche il limite inferiore rimane positivo (**+0,0112**).

→ Il bootstrap fornisce quindi evidenza che il miglioramento di Accuracy osservato con il modello Augmented è **robusto rispetto al ricampionamento del test set**. In questo esperimento, l'aggiunta delle immagini sintetiche è associata a un miglioramento dell'Accuracy rispetto al Baseline e la differenza osservata non sembra essere dovuta semplicemente alla variabilità del campione di test.

## 15. Error Analysis

Le metriche aggregate (accuracy, F1, ecc.) dicono se in media l'Augmented va meglio del Baseline, ma non dicono **su quali immagini specifiche** cambia qualcosa. Qui vengono guardate direttamente le singole predizioni sul test set, appaiate tra i due modelli (stesso test set, stesso ordine), per trovare due categorie di casi:

- **Corretti dall'Augmented**: il Baseline sbagliava, l'Augmented ci azzecca — la prova diretta di un beneficio concreto, non solo statistico.
- **Rotti dall'Augmented**: il Baseline ci azzeccava, l'Augmented sbaglia — il caso opposto, altrettanto importante da non nascondere.

In [ ]:
# ============================================================
# Error Analysis: conteggio dei casi che cambiano tra i due modelli
# ============================================================
# baseline_results e augmented_results derivano dalla stessa chiamata
# evaluate(test_loader), quindi le etichette sono nello stesso ordine:
# posso confrontare le predizioni indice per indice.
assert np.array_equal(baseline_results["labels"], augmented_results["labels"])

baseline_correct = baseline_results["preds"] == baseline_results["labels"]
augmented_correct = augmented_results["preds"] == augmented_results["labels"]

fixed_idx = np.where((~baseline_correct) & augmented_correct)[0]   # baseline sbaglia, augmented giusto
broken_idx = np.where(baseline_correct & (~augmented_correct))[0]  # baseline giusto, augmented sbaglia
both_wrong_idx = np.where((~baseline_correct) & (~augmented_correct))[0]

print(f"Corretti dall'Augmented (baseline sbaglia -> augmented giusto): {len(fixed_idx)}")
print(f"Sbagliati dall'Augmented (baseline giusto -> augmented sbaglia):    {len(broken_idx)}")
print(f"Sbagliati da entrambi:                                         {len(both_wrong_idx)}")
print(f"Bilancio netto (corretti - sbagliati): {len(fixed_idx) - len(broken_idx)}")

In [ ]:
# ============================================================
# Esempi visivi dei casi che cambiano tra Baseline e Augmented
# ============================================================
# Uso test_raw (senza trasformazioni) per mostrare le foto originali,
# indicizzabile allo stesso modo di test_dataset perché entrambi
# caricano lo stesso split "test" nello stesso ordine.
def show_error_examples(indices, title, n=6):
    n = min(n, len(indices))
    if n == 0:
        print(f"{title}: nessun caso trovato.")
        return
    sample = np.random.default_rng(SEED).choice(indices, size=n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.8))
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, sample):
        img, _ = test_raw[idx]
        true_label = CLASSES[baseline_results["labels"][idx]].replace("_", " ")
        base_pred = CLASSES[baseline_results["preds"][idx]].replace("_", " ")
        aug_pred = CLASSES[augmented_results["preds"][idx]].replace("_", " ")
        ax.imshow(img)
        ax.set_title(f"vero: {true_label}\nbase: {base_pred}\naug: {aug_pred}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_error_examples(fixed_idx, "Baseline sbaglia -> Augmented corregge")
show_error_examples(broken_idx, "Baseline giusto -> Augmented sbaglia")

### Analisi degli errori — cosa cambia tra Baseline e Augmented

Per capire meglio l'effetto delle immagini sintetiche, vengono confrontate le predizioni dei due modelli **sulle stesse immagini del test set**.

I risultati mostrano che:

- **132 immagini** sbagliate dal Baseline vengono invece classificate correttamente dall'Augmented;
- **63 immagini** corrette dal Baseline vengono sbagliate dall'Augmented;
- **285 immagini** vengono sbagliate da entrambi i modelli;
- il bilancio complessivo è quindi di **+69 immagini corrette** a favore dell'Augmented.

Questo risultato fa vedere che il miglioramento dell'Augmented non significa che tutte le predizioni siano migliori: alcune immagini vengono recuperate grazie al nuovo training, mentre altre che il Baseline classificava correttamente vengono perse.

Il bilancio è comunque positivo, perché le **132 correzioni sono maggiori dei 63 nuovi errori**.

### Osservazione di alcuni esempi

[Da completare dopo l'esecuzione: guardare la griglia di esempi visivi appena generata e descrivere 2-3 casi in cui il Baseline sbaglia e l'Augmented corregge (razze confuse tra loro), e 1-2 casi opposti in cui l'Augmented introduce un errore che il Baseline non commetteva. I nomi di razza riportati in una versione precedente di questo commento si riferivano a un'esecuzione diversa e non sono più validi.]

→ L'aggiunta delle immagini sintetiche **modifica quindi il comportamento del modello**, migliorando alcune decisioni ma peggiorandone altre. Nel complesso l'effetto è positivo, con **69 classificazioni corrette in più**, coerentemente con l'aumento di Accuracy osservato nel confronto tra Baseline e Augmented.

## 16. Conclusioni e limiti

### Riepilogo del flusso di lavoro

```
Oxford-IIIT Pet (37 classi, trainval + test)
    |  split stratificato 80/20 con train_test_split
    v
Training (80% del trainval)  +  Validation (20% del trainval)  +  Test reale (intoccato)
    |
    |-- Esperimento A: training diretto -----------------------------> Baseline
    |
    |-- Campiono il 30% del training (stratificato)
    |       |
    |       v
    |   Captioning (BLIP) -> Generazione testo (Qwen2.5-1.5B-Instruct) -> Generazione immagini (sdxl-turbo)
    |       |
    |       v
    |   Dataset sintetico, il 30% del training (etichettato con la classe reale di origine)
    |       |
    |       v
    |-- Training + sintetico -------------------------------> Esperimento B: Augmented
    |
    v
Valutazione di entrambi sullo stesso test set: accuracy, top-3 accuracy, precision/recall/F1,
matrice di confusione, delta di recall per classe, intervallo di confidenza bootstrap,
error analysis caso per caso
```

### Risultati e considerazioni finali

L'obiettivo del progetto era verificare se l'introduzione di **immagini sintetiche generate tramite una pipeline di Data Augmentation generativa** potesse migliorare la capacità di generalizzazione del classificatore su immagini reali.

Per verificarlo sono stati confrontati due modelli con la stessa architettura e la stessa configurazione di training:

- **Baseline**, addestrato utilizzando solamente le immagini reali;
- **Augmented**, addestrato utilizzando le stesse immagini reali insieme alle immagini sintetiche generate dalla pipeline **BLIP → Qwen2.5-1.5B-Instruct → SDXL-Turbo**.

Sul test set reale, il modello **Augmented ottiene risultati migliori del Baseline su tutte le principali metriche considerate**.

L'Accuracy passa da **88,66% a 90,54%**, con un miglioramento di **+1,88 punti percentuali**. Si osservano miglioramenti anche nella **Precision macro (+1,86 punti)**, nel **Recall macro (+1,89 punti)** e nell'**F1-score macro (+1,96 punti)**.

L'intervallo di confidenza bootstrap calcolato sulla differenza di Accuracy è **[+1,12; +2,64] punti percentuali** e non comprende lo zero. Questo risultato fornisce quindi ulteriore evidenza che, su questo test set, il miglioramento osservato non sia semplicemente riconducibile alla variabilità del campione.

Il beneficio dell'augmentation non è però identico per tutte le **37 classi**. Il recall migliora per **24 classi**, diminuisce per **5 classi** e rimane invariato per **8 classi**.

Tra i miglioramenti più evidenti troviamo `Maine Coon`, il cui recall passa dal **77% all'86%**, e `American Pit Bull Terrier`, che passa dal **42% al 50%**.

Anche l'analisi delle singole predizioni mostra un vantaggio complessivo per il modello Augmented: **132 immagini classificate erroneamente dal Baseline vengono classificate correttamente dall'Augmented**, mentre **63 immagini corrette nel Baseline diventano errate nell'Augmented**. Il bilancio complessivo è quindi di **+69 classificazioni corrette** a favore del modello Augmented.

Nel complesso, l'esperimento mostra che, **in questa configurazione**, l'aggiunta di dati sintetici ha prodotto un miglioramento misurabile delle prestazioni del classificatore sui dati reali.

Il **CLIP score medio di 35,16** fornisce un'indicazione della coerenza semantica tra prompt e immagini sintetiche generate, mentre il confronto tra Baseline e Augmented permette di valutarne l'effetto effettivo sul task di classificazione.

L'aspetto più importante emerso dall'esperimento è quindi che la **Data Augmentation generativa può rappresentare uno strumento utile per arricchire il training set**, ma il suo contributo deve essere verificato sperimentalmente sul test set reale e analizzato sia attraverso metriche globali sia osservando il comportamento delle singole classi.